# Four Routes to Default Probability: Biocon Limited

**Financial Risk Analytics and Management**  
**Company:** Biocon Limited (NSE: BIOCON)  
**Main valuation date:** 30 September 2026

This notebook is the complete reproducible submission. It starts with the five consolidated raw workbooks, preserves valuation-date information cutoffs, calculates every required model, runs the independent checks, and ends with the Part E comparison.

## Research design

The assignment compares four conceptually different routes to default risk:

1. **Part A, structural market route:** option-implied equity volatility is converted into firm asset value and asset volatility using the Merton model.
2. **Part B, forecast volatility routes:** Student_t GARCH and Random Forest forecasts replace option-implied volatility in the same structural framework.
3. **Part C, rating route:** the CRISIL one-year default rate is converted to distance to default and reverse-solved into the volatility required by Merton.
4. **Part D, accounting route:** Altman Z, Z_double_prime and emerging-market scores are calculated from consolidated financial statements; the EM rating equivalent is mapped to a PD and reverse-solved through Merton.

Part E compares the routes without treating them as interchangeable. Market measures are point_in_time, ratings are primarily through_the_cycle, and Altman is an accounting point_in_time measure.

## Inputs and reproducibility

Upload exactly these five files when prompted:

1. `Biocon_data_updated.xlsx`
2. `Financial_Statements (2) (1).xlsx`
3. `Biocon_options-_data.xlsx`
4. `Auctions of 91-Day Government of India Treasury Bills.xlsx`
5. `Auctions of 364-Day Government of India Treasury Bills.xlsx`

The market workbook contains Biocon prices and adjusted returns, Nifty 50, India VIX, result months, dated shares outstanding and dividends. The financial workbook contains annual statements, quarterly balance sheets, rating inputs and the Altman rating map. Options and both exact-tenor RBI auction series remain separate raw workbooks.

All monetary statement values are in **INR million** unless stated otherwise. Published T_bill yields remain in percent in the raw tables and are converted using `r = ln(1 + R/100)`.

In [ ]:
%pip install -q arch==8.0.0 statsmodels==0.15.0 scikit-learn==1.9.1 py-vollib==1.0.12 merton==1.0.2 openpyxl==3.1.5

In [ ]:
from pathlib import Path
import sys

try:
    from google.colab import files
except ImportError as exc:
    raise RuntimeError('Run this upload cell in Google Colab.') from exc

expected_files = {
    'Biocon_data_updated.xlsx',
    'Financial_Statements (2) (1).xlsx',
    'Biocon_options-_data.xlsx',
    'Auctions of 91-Day Government of India Treasury Bills.xlsx',
    'Auctions of 364-Day Government of India Treasury Bills.xlsx',
}
uploaded = files.upload()
missing = expected_files.difference(uploaded)
assert not missing, f'Missing required files: {sorted(missing)}'

Path('src').mkdir(exist_ok=True)
Path('data/processed').mkdir(parents=True, exist_ok=True)
Path('outputs/figures').mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(Path.cwd() / 'src'))
print('All five input files are available.')

## Embedded model code

The next cells write the audited modules bundled inside this notebook. This keeps the submission self-contained while retaining short, separately testable model sections. The modules do not download or silently replace data.

In [ ]:
%%writefile src/input_data.py
"""Locate the five assignment input workbooks locally or in Google Colab."""
import os
from pathlib import Path


ROOT = Path(__file__).resolve().parents[1]
LOCAL_INPUTS = ROOT / 'data/raw/input_files'

MARKET_FILE = 'Biocon_data_updated.xlsx'
FINANCIAL_FILE = 'Financial_Statements (2) (1).xlsx'
OPTIONS_FILE = 'Biocon_options-_data.xlsx'
RATE_91_FILE = 'Auctions of 91-Day Government of India Treasury Bills.xlsx'
RATE_364_FILE = 'Auctions of 364-Day Government of India Treasury Bills.xlsx'


def file(name):
    """Return an input path from the project, Colab, cwd or an override folder."""
    folders = [
        Path(os.environ['FRAM_INPUT_DIR']) if os.getenv('FRAM_INPUT_DIR') else None,
        LOCAL_INPUTS,
        Path.cwd(),
        Path('/content'),
        Path.home() / 'Desktop',
    ]
    for folder in filter(None, folders):
        candidate = folder / name
        if candidate.exists():
            return candidate
    raise FileNotFoundError(f'Missing input file: {name}')


def all_files():
    names = [MARKET_FILE, FINANCIAL_FILE, OPTIONS_FILE,
             RATE_91_FILE, RATE_364_FILE]
    return {name: file(name) for name in names}

In [ ]:
%%writefile src/preprocess_market_inputs.py
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import numpy as np
import pandas as pd

import input_data

ROOT = Path(__file__).resolve().parents[1]
OUT = ROOT / "data" / "processed"
FIGURE_OUT = ROOT / "outputs" / "figures"
OUT.mkdir(parents=True, exist_ok=True)
FIGURE_OUT.mkdir(parents=True, exist_ok=True)

MARKET_FILE = input_data.file(input_data.MARKET_FILE)
RATE_91_FILE = input_data.file(input_data.RATE_91_FILE)
RATE_364_FILE = input_data.file(input_data.RATE_364_FILE)
OPTIONS_FILE = input_data.file(input_data.OPTIONS_FILE)

DIVIDEND_EVENTS = pd.read_excel(MARKET_FILE, sheet_name="Dividends").rename(
    columns={"dividend_per_share_inr": "dividend_per_share"}
)
DIVIDEND_EVENTS["ex_date"] = pd.to_datetime(DIVIDEND_EVENTS["ex_date"])

VALUATION_DATES = pd.to_datetime([
    "2025-10-31", "2025-11-28", "2025-12-31", "2026-01-30",
    "2026-02-27", "2026-03-30", "2026-04-30", "2026-05-29",
    "2026-06-30", "2026-07-31", "2026-08-31", "2026-09-30",
])

YEAR_ENDS = pd.to_datetime([
    "2022-03-31", "2023-03-31", "2024-03-31",
    "2025-03-31", "2026-03-31",
])


def to_number(series):
    return pd.to_numeric(
        series.astype(str).str.replace(",", "", regex=False),
        errors="coerce",
    )


def latest_on_or_before(data, dates, columns):
    targets = pd.DataFrame({"target_date": dates.astype("datetime64[ns]")})
    source = data[["date", *columns]].dropna(subset=["date"]).copy()
    source["date"] = pd.to_datetime(source["date"]).astype("datetime64[ns]")

    result = pd.merge_asof(
        targets.sort_values("target_date"),
        source.sort_values("date"),
        left_on="target_date",
        right_on="date",
        direction="backward",
    )
    result["lag_calendar_days"] = (result["target_date"] - result["date"]).dt.days
    return result


def load_equity():
    data = pd.read_excel(MARKET_FILE, sheet_name="Biocon")
    data.columns = data.columns.str.strip()
    data = data.rename(columns={
        "Symbol": "symbol",
        "Series": "series",
        "Date": "date",
        "Prev Close": "previous_close",
        "Open Price": "open_price",
        "High Price": "high_price",
        "Low Price": "low_price",
        "Last Price": "last_price",
        "Close Price": "close_price",
        "Average Price": "average_price",
        "Total Traded Quantity": "volume",
        "Turnover ₹": "turnover_inr",
        "No. of Trades": "number_of_trades",
        "Deliverable Qty": "deliverable_quantity",
        "% Dly Qt to Traded Qty": "deliverable_percentage",
        "Adjusted Prices": "adjusted_price",
        "Adjusted Returns": "adjusted_return",
    })

    data["symbol"] = data["symbol"].astype(str).str.strip()
    data["series"] = data["series"].astype(str).str.strip()
    data["date"] = pd.to_datetime(data["date"], errors="raise")
    data = data[(data["symbol"] == "BIOCON") & (data["series"] == "EQ")].copy()

    numeric_columns = [
        "previous_close", "open_price", "high_price", "low_price",
        "last_price", "close_price", "average_price", "volume",
        "turnover_inr", "number_of_trades", "deliverable_quantity",
        "deliverable_percentage", "adjusted_price", "adjusted_return",
    ]
    for column in numeric_columns:
        data[column] = to_number(data[column])

    data = data.sort_values("date").drop_duplicates("date", keep="last").reset_index(drop=True)
    data["calculated_adjusted_return"] = np.log(
        data["adjusted_price"] / data["adjusted_price"].shift(1)
    )
    data["return_difference"] = (
        data["adjusted_return"] - data["calculated_adjusted_return"]
    )
    data["model_return"] = data["adjusted_return"]
    return data


def load_rate(file, maturity):
    raw = pd.read_excel(file, header=None)
    rates = raw.iloc[:, [1, 13]].copy()
    rates.columns = ["date", "yield_percent"]
    rates["date"] = pd.to_datetime(rates["date"], format="mixed", errors="coerce")
    rates["yield_percent"] = pd.to_numeric(rates["yield_percent"], errors="coerce")
    rates = rates.dropna()
    rates = rates[rates["yield_percent"] > 0]
    rates = rates.sort_values("date").drop_duplicates("date", keep="last")
    rates["maturity_days"] = maturity
    rates["source_file"] = file.name
    return rates.reset_index(drop=True)


# -----------------------------------------------------------------------------
# OPTIONS PREPROCESSING
# -----------------------------------------------------------------------------
def trailing_dividend_per_share(valuation_date):
    """Return cash dividends with ex-dates in the trailing 12 months."""
    window_start = valuation_date - pd.DateOffset(years=1)
    in_window = DIVIDEND_EVENTS["ex_date"].gt(window_start) & DIVIDEND_EVENTS["ex_date"].le(valuation_date)
    return DIVIDEND_EVENTS.loc[in_window, "dividend_per_share"].sum()


def preprocess_options(valuation_rates, valuation_prices):
    """Apply the assignment's option-chain cleaning rules for BSM."""
    workbook = pd.ExcelFile(OPTIONS_FILE)
    frames = []

    for sheet in workbook.sheet_names:
        frame = pd.read_excel(OPTIONS_FILE, sheet_name=sheet)
        frame.columns = frame.columns.str.strip()
        frame.insert(0, "source_sheet", sheet)
        frame.insert(1, "source_row", np.arange(2, len(frame) + 2))
        frames.append(frame)

    raw = pd.concat(frames, ignore_index=True)
    raw = raw.rename(columns={
        "TradDt": "valuation_date",
        "FinInstrmTp": "instrument_type",
        "FinInstrmId": "instrument_id",
        "TckrSymb": "ticker",
        "XpryDt": "expiry_date",
        "StrkPric": "strike_price",
        "OptnTp": "option_type",
        "FinInstrmNm": "contract_name",
        "ClsPric": "market_price",
        "UndrlygPric": "spot_price",
        "SttlmPric": "settlement_price",
        "OpnIntrst": "open_interest",
        "TtlTradgVol": "traded_volume",
        "NewBrdLotQty": "lot_size",
    })

    raw["valuation_date"] = pd.to_datetime(raw["valuation_date"], errors="coerce")
    raw["expiry_date"] = pd.to_datetime(raw["expiry_date"], errors="coerce")
    numeric_columns = [
        "strike_price", "market_price", "spot_price", "settlement_price",
        "open_interest", "traded_volume", "lot_size",
    ]
    for column in numeric_columns:
        raw[column] = pd.to_numeric(raw[column], errors="coerce")

    raw["days_to_expiry"] = (raw["expiry_date"] - raw["valuation_date"]).dt.days
    # Section 2.2 requires settlement prices for IV.
    raw["option_close_price"] = raw["market_price"]
    raw["market_price"] = raw["settlement_price"]
    raw["time_to_expiry_years"] = raw["days_to_expiry"] / 365
    raw["moneyness_strike_to_spot"] = raw["strike_price"] / raw["spot_price"]

    # Use the near month when it has more than seven days left; otherwise use
    # the next listed monthly expiry, as required by the assignment.
    is_option = raw["instrument_type"].eq("STO")
    selected_expiries = {}
    for valuation_date, group in raw.loc[is_option].groupby("valuation_date"):
        expiries = sorted(group.loc[group["expiry_date"] > valuation_date, "expiry_date"].dropna().unique())
        if not expiries:
            selected_expiries[valuation_date] = pd.NaT
            continue
        near_month = pd.Timestamp(expiries[0])
        selected_expiries[valuation_date] = (
            near_month
            if (near_month - valuation_date).days > 7
            else (pd.Timestamp(expiries[1]) if len(expiries) > 1 else pd.NaT)
        )
    raw["selected_expiry_date"] = raw["valuation_date"].map(selected_expiries)

    # Attach the assignment risk-free rate and trailing dividend yield.
    bsm_rates = valuation_rates.loc[
        valuation_rates["maturity_days"].eq(91),
        ["target_date", "continuous_rate"],
    ].rename(columns={
        "target_date": "valuation_date",
        "continuous_rate": "risk_free_rate",
    })
    raw = raw.merge(bsm_rates, on="valuation_date", how="left")
    raw["ttm_dividend_per_share"] = raw["valuation_date"].map(trailing_dividend_per_share)
    raw["dividend_yield"] = raw["ttm_dividend_per_share"] / raw["spot_price"]

    discounted_spot = raw["spot_price"] * np.exp(
        -raw["dividend_yield"] * raw["time_to_expiry_years"]
    )
    discounted_strike = raw["strike_price"] * np.exp(
        -raw["risk_free_rate"] * raw["time_to_expiry_years"]
    )
    raw["no_arbitrage_lower_bound"] = np.where(
        raw["option_type"].eq("CE"),
        np.maximum(discounted_spot - discounted_strike, 0),
        np.where(
            raw["option_type"].eq("PE"),
            np.maximum(discounted_strike - discounted_spot, 0),
            np.nan,
        ),
    )
    raw["no_arbitrage_upper_bound"] = np.where(
        raw["option_type"].eq("CE"),
        discounted_spot,
        np.where(raw["option_type"].eq("PE"), discounted_strike, np.nan),
    )
    raw["exclusion_reason"] = ""

    def add_reason(mask, reason):
        separator = np.where(raw.loc[mask, "exclusion_reason"].eq(""), "", "; ")
        raw.loc[mask, "exclusion_reason"] += separator + reason

    add_reason(~is_option, "futures contract")
    add_reason(is_option & ~raw["option_type"].isin(["CE", "PE"]), "invalid option type")
    add_reason(is_option & raw["strike_price"].le(0), "missing or non-positive strike")
    add_reason(is_option & raw["spot_price"].le(0), "missing or non-positive spot")
    add_reason(is_option & raw["market_price"].le(0), "missing or non-positive settlement price")
    add_reason(
        is_option & raw["expiry_date"].ne(raw["selected_expiry_date"]),
        "not assignment-selected expiry",
    )
    add_reason(is_option & raw["traded_volume"].fillna(0).le(0), "no trading volume")
    add_reason(
        is_option & ~raw["moneyness_strike_to_spot"].between(0.90, 1.10, inclusive="both"),
        "outside 0.90 to 1.10 moneyness",
    )
    add_reason(
        is_option & raw["market_price"].lt(raw["no_arbitrage_lower_bound"] - 1e-9),
        "below no-arbitrage lower bound",
    )
    # The upper bound is retained as an additional diagnostic. Section 2.2 only
    # mandates deletion for lower-bound violations.
    raw["upper_bound_violation"] = (
        is_option & raw["market_price"].gt(raw["no_arbitrage_upper_bound"] + 1e-9)
    )

    raw["included_for_bsm"] = raw["exclusion_reason"].eq("")
    clean_columns = [
        "source_sheet", "source_row", "valuation_date", "expiry_date",
        "instrument_id", "contract_name", "option_type", "strike_price",
        "spot_price", "market_price", "option_close_price", "settlement_price", "traded_volume",
        "open_interest", "lot_size", "days_to_expiry", "time_to_expiry_years",
        "moneyness_strike_to_spot", "risk_free_rate", "ttm_dividend_per_share", "dividend_yield",
        "no_arbitrage_lower_bound", "no_arbitrage_upper_bound",
        "upper_bound_violation",
    ]
    clean = raw.loc[raw["included_for_bsm"], clean_columns].copy()
    clean = clean.sort_values(
        ["valuation_date", "expiry_date", "option_type", "strike_price"]
    ).reset_index(drop=True)

    exclusions = raw.loc[~raw["included_for_bsm"], [
        "source_sheet", "source_row", "valuation_date", "instrument_type",
        "instrument_id", "contract_name", "option_type", "expiry_date",
        "selected_expiry_date", "strike_price", "spot_price", "market_price",
        "traded_volume", "moneyness_strike_to_spot", "risk_free_rate",
        "ttm_dividend_per_share", "dividend_yield", "no_arbitrage_lower_bound", "no_arbitrage_upper_bound",
        "upper_bound_violation",
        "exclusion_reason",
    ]].copy()

    summary = raw.groupby("valuation_date").agg(
        source_rows=("instrument_id", "size"),
        futures_rows=("instrument_type", lambda x: x.eq("STF").sum()),
        option_rows=("instrument_type", lambda x: x.eq("STO").sum()),
        selected_expiry_options=("expiry_date", lambda x: (x == raw.loc[x.index, "selected_expiry_date"]).sum()),
        zero_volume_options=("traded_volume", lambda x: ((x.fillna(0) <= 0) & is_option.loc[x.index]).sum()),
        within_moneyness_options=("moneyness_strike_to_spot", lambda x: (x.between(0.90, 1.10, inclusive="both") & is_option.loc[x.index]).sum()),
        bsm_ready_options=("included_for_bsm", "sum"),
    ).reset_index()

    # Sequential attrition required by the assignment data appendix.
    audit_rows = []
    for valuation_date, date_rows in raw.groupby("valuation_date"):
        current = date_rows["instrument_type"].eq("STO")
        start = int(current.sum())
        audit_rows.append([valuation_date, "Raw stock options", 0, start])
        rules = [
            ("Wrong expiry", date_rows["expiry_date"].eq(date_rows["selected_expiry_date"])),
            ("Zero volume", date_rows["traded_volume"].fillna(0).gt(0)),
            ("Outside 0.90 to 1.10 moneyness", date_rows["moneyness_strike_to_spot"].between(0.90, 1.10, inclusive="both")),
            ("Below no-arbitrage lower bound", date_rows["market_price"].ge(date_rows["no_arbitrage_lower_bound"] - 1e-9)),
        ]
        for step, passes in rules:
            dropped = int((current & ~passes).sum())
            current &= passes
            audit_rows.append([valuation_date, step, dropped, int(current.sum())])
        audit_rows.append([
            valuation_date,
            "Above upper bound (diagnostic only)",
            int((current & date_rows["upper_bound_violation"]).sum()),
            int(current.sum()),
        ])
    cleaning_audit = pd.DataFrame(
        audit_rows, columns=["valuation_date", "step", "dropped_at_step", "remaining"]
    )
    total_rows = []
    for step, group in cleaning_audit.groupby("step", sort=False):
        total_rows.append(["TOTAL", step, int(group["dropped_at_step"].sum()), int(group["remaining"].sum())])
    cleaning_audit = pd.concat([
        cleaning_audit,
        pd.DataFrame(total_rows, columns=cleaning_audit.columns),
    ], ignore_index=True)

    option_spot = raw.loc[is_option].groupby("valuation_date", as_index=False).agg(
        option_spot=("spot_price", "first"),
        option_spot_min=("spot_price", "min"),
        option_spot_max=("spot_price", "max"),
    )
    spot_reconciliation = option_spot.merge(
        valuation_prices[["target_date", "close_price"]].rename(
            columns={"target_date": "valuation_date", "close_price": "equity_close"}
        ),
        on="valuation_date",
        how="left",
    )
    spot_reconciliation["option_minus_equity"] = (
        spot_reconciliation["option_spot"] - spot_reconciliation["equity_close"]
    )

    assert len(raw) == 1541, "Raw F&O row count changed"
    assert is_option.sum() == 1505 and (~is_option).sum() == 36, "Instrument counts changed"
    assert raw["valuation_date"].nunique() == 12, "Expected all 12 valuation dates"
    assert not raw.duplicated(["valuation_date", "instrument_id"]).any(), "Duplicate contracts found"
    assert clean["risk_free_rate"].notna().all(), "Missing BSM rate"
    assert clean["ttm_dividend_per_share"].eq(0.50).all(), "Unexpected TTM dividend"
    assert not clean.duplicated(["valuation_date", "instrument_id"]).any(), "Duplicate clean contracts"
    total_audit = cleaning_audit.loc[cleaning_audit["valuation_date"].eq("TOTAL")]
    assert int(total_audit.loc[total_audit["step"].eq("Below no-arbitrage lower bound"), "remaining"].iloc[0]) == len(clean)

    return raw, clean, exclusions, summary, cleaning_audit, spot_reconciliation


def save_line_chart(data, x, y, title, y_label, file_name, color, note=None):
    fig, ax = plt.subplots(figsize=(12, 5.5))
    ax.plot(data[x], data[y], color=color, linewidth=1.25)
    if data[y].min() < 0 < data[y].max():
        ax.axhline(0, color="#64748b", linewidth=0.8, alpha=0.7)
    ax.set_title(title, fontsize=14, fontweight="bold", pad=12)
    ax.set_xlabel("Date")
    ax.set_ylabel(y_label)
    ax.grid(axis="y", color="#cbd5e1", linewidth=0.6, alpha=0.7)
    ax.spines[["top", "right"]].set_visible(False)
    ax.xaxis.set_major_locator(mdates.AutoDateLocator())
    ax.xaxis.set_major_formatter(mdates.ConciseDateFormatter(ax.xaxis.get_major_locator()))
    if note:
        fig.text(0.01, 0.01, note, fontsize=9, color="#475569")
    fig.tight_layout()
    fig.savefig(FIGURE_OUT / file_name, dpi=200, bbox_inches="tight")
    # Keep the figure open so VS Code displays it after processing.


def create_charts(equity, all_rates):
    save_line_chart(
        equity,
        "date",
        "adjusted_price",
        "Biocon Corporate-Action-Adjusted Share Price (10 Years)",
        "Adjusted price (INR)",
        "biocon_adjusted_price_10_years.png",
        "#7c3aed",
        "Source: user-supplied Adjusted Prices; adjusted for applicable corporate actions.",
    )

    returns = equity[["date", "model_return"]].dropna().copy()
    returns["return_percent"] = returns["model_return"] * 100
    save_line_chart(
        returns,
        "date",
        "return_percent",
        "Biocon Adjusted Daily Returns (10 Years)",
        "Adjusted return (%)",
        "biocon_adjusted_returns_10_years.png",
        "#2563eb",
    )

    required_period = all_rates[
        all_rates["date"].between(VALUATION_DATES.min(), VALUATION_DATES.max())
    ]
    for maturity, color in [(91, "#0f766e"), (364, "#b45309")]:
        rates = required_period[required_period["maturity_days"] == maturity]
        save_line_chart(
            rates,
            "date",
            "yield_percent",
            f"RBI {maturity}-Day Treasury Bill Yield",
            "Implicit yield at cut-off price (%)",
            f"rbi_{maturity}_day_tbill_rates_required_period.png",
            color,
        )


def main():
    # Load and validate the user-supplied equity history.
    equity = load_equity()

    valuation_prices = latest_on_or_before(
        equity, VALUATION_DATES, ["close_price", "adjusted_price"]
    )
    year_end_prices = latest_on_or_before(
        equity, YEAR_ENDS, ["close_price", "adjusted_price"]
    )
    year_end_prices.insert(0, "fiscal_year", [2022, 2023, 2024, 2025, 2026])

    # Load the 91-day and 364-day RBI auction series.
    rate_91 = load_rate(RATE_91_FILE, 91)
    rate_364 = load_rate(RATE_364_FILE, 364)
    all_rates = pd.concat([rate_91, rate_364], ignore_index=True)

    # Match each valuation date to the latest available auction rate.
    selected_rates = []
    for maturity, group in all_rates.groupby("maturity_days"):
        selected = latest_on_or_before(group, VALUATION_DATES, ["yield_percent"])
        selected["maturity_days"] = maturity
        selected["simple_rate_decimal"] = selected["yield_percent"] / 100
        selected["continuous_rate"] = np.log1p(selected["simple_rate_decimal"])
        selected_rates.append(selected)

    valuation_rates = pd.concat(selected_rates, ignore_index=True)
    valuation_rates = valuation_rates.sort_values(["target_date", "maturity_days"])

    # -------------------------------------------------------------------------
    # OPTIONS PREPROCESSING
    # -------------------------------------------------------------------------
    (
        options_raw,
        options_clean,
        option_exclusions,
        options_summary,
        options_cleaning_audit,
        option_spot_reconciliation,
    ) = preprocess_options(
        valuation_rates, valuation_prices
    )

    # Save all four charts and display them together on screen.
    create_charts(equity, all_rates)
    plt.show()

    # Save the cleaned datasets used by the models.
    equity.to_csv(OUT / "biocon_equity_adjusted.csv", index=False, date_format="%Y-%m-%d")
    valuation_prices.to_csv(OUT / "valuation_date_equity_prices.csv", index=False, date_format="%Y-%m-%d")
    year_end_prices.to_csv(OUT / "fiscal_year_end_equity_prices.csv", index=False, date_format="%Y-%m-%d")
    all_rates.to_csv(OUT / "rbi_tbill_auction_yields.csv", index=False, date_format="%Y-%m-%d")
    valuation_rates.to_csv(OUT / "valuation_date_tbill_rates.csv", index=False, date_format="%Y-%m-%d")
    options_raw.to_csv(OUT / "options_all_contracts.csv", index=False, date_format="%Y-%m-%d")
    options_clean.to_csv(OUT / "options_bsm_ready.csv", index=False, date_format="%Y-%m-%d")
    option_exclusions.to_csv(OUT / "options_exclusion_log.csv", index=False, date_format="%Y-%m-%d")
    options_summary.to_csv(OUT / "options_preprocessing_summary.csv", index=False, date_format="%Y-%m-%d")
    options_cleaning_audit.to_csv(OUT / "options_sequential_cleaning_audit.csv", index=False, date_format="%Y-%m-%d")
    option_spot_reconciliation.to_csv(OUT / "option_spot_reconciliation.csv", index=False, date_format="%Y-%m-%d")

    print(f"Equity observations processed: {len(equity)}")

    print(f"Duplicate dates: {equity['date'].duplicated().sum()}")
    print(f"91-day rate observations: {len(rate_91)}")
    print(f"364-day rate observations: {len(rate_364)}")
    print(f"Raw option-chain rows: {len(options_raw)}")
    print(f"BSM-ready option rows: {len(options_clean)}")
    print(f"Excluded option/futures rows: {len(option_exclusions)}")
    print(f"Processed data saved to: {OUT}")
    print(f"Charts saved to: {FIGURE_OUT}")

    print("\nFiscal year-end prices")
    print(year_end_prices.to_string(index=False))

    print("\nValuation-date equity prices")
    print(valuation_prices.to_string(index=False))

    print("\nValuation-date T-bill rates")
    print(valuation_rates.to_string(index=False))

    print("\nOptions preprocessing summary")
    print(options_summary.to_string(index=False))

    print("\nSequential options cleaning audit (total)")
    print(options_cleaning_audit.loc[
        options_cleaning_audit["valuation_date"].eq("TOTAL")
    ].to_string(index=False))

    print("\nOption spot reconciliation")
    print(option_spot_reconciliation.to_string(index=False))


if __name__ == "__main__":
    main()

In [ ]:
%%writefile src/implied_volatility.py
"""Part A: own Newton-Raphson solver and assignment ATM interpolation."""
from pathlib import Path
from math import erf, exp, log, sqrt, pi, isfinite
import warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# The assignment names py_vollib as a check-only reference implementation.
with warnings.catch_warnings():
    warnings.filterwarnings('ignore', message='py_vollib is deprecated.*')
    from py_vollib.black_scholes_merton.implied_volatility import (
        implied_volatility as py_vollib_iv,
    )

ROOT = Path(__file__).resolve().parents[1]
TOL = 0.0001
REFERENCE_IV_TOL = 0.00001


def bsm(s, k, t, r, q, sigma, kind):
    if kind not in {'CE', 'PE'}:
        raise ValueError(f'Unsupported option type: {kind}')
    d1 = (log(s / k) + (r - q + sigma * sigma / 2) * t) / (sigma * sqrt(t))
    d2 = d1 - sigma * sqrt(t)
    cdf = lambda x: (1 + erf(x / sqrt(2))) / 2
    a, b = s * exp(-q * t), k * exp(-r * t)
    price = a * cdf(d1) - b * cdf(d2) if kind == 'CE' else b * cdf(-d2) - a * cdf(-d1)
    vega = a * sqrt(t) * exp(-d1 * d1 / 2) / sqrt(2 * pi)
    return price, vega


def solve_iv(s, k, t, r, q, price, kind):
    if kind not in {'CE', 'PE'}:
        raise ValueError(f'Unsupported option type: {kind}')
    if not all(isfinite(x) for x in [s, k, t, r, q, price]) or min(s, k, t, price) <= 0:
        return np.nan, 'invalid_input', 0, np.nan
    sigma = 0.3
    for n in range(1, 101):
        fitted, vega = bsm(s, k, t, r, q, sigma, kind)
        error = fitted - price
        if abs(error) < TOL:
            return sigma, 'newton', n, error
        if vega < 1e-10:
            break
        candidate = sigma - error / vega
        if not 0.01 < candidate < 3:
            break
        sigma = candidate
    # Own bisection fallback, without a library IV solver.
    lo, hi = 0.01, 3.0
    flo = bsm(s, k, t, r, q, lo, kind)[0] - price
    fhi = bsm(s, k, t, r, q, hi, kind)[0] - price
    for edge, error in [(lo, flo), (hi, fhi)]:
        if abs(error) < TOL:
            return edge, 'bisection', n, error
    if flo * fhi > 0:
        return np.nan, 'no_solution_in_bracket', n, np.nan
    for j in range(1, 201):
        sigma = (lo + hi) / 2
        error = bsm(s, k, t, r, q, sigma, kind)[0] - price
        if abs(error) < TOL:
            return sigma, 'bisection', n + j, error
        if error > 0:
            hi = sigma
        else:
            lo = sigma
    return np.nan, 'not_converged', n + 200, error


def reference_iv(s, k, t, r, q, price, kind):
    """Return py_vollib IV for validation only; never use it as model input."""
    flag = {'CE': 'c', 'PE': 'p'}[kind]
    try:
        return py_vollib_iv(price, s, k, t, r, q, flag)
    except Exception:
        return np.nan


def atm(group, iv_column='iv'):
    group = group.dropna(subset=[iv_column]).sort_values('moneyness_strike_to_spot')
    x = group.moneyness_strike_to_spot
    exact = group.loc[x.eq(1)]
    if len(exact):
        return float(exact[iv_column].iloc[0])
    below, above = group.loc[x < 1], group.loc[x > 1]
    if below.empty or above.empty:
        return np.nan  # Never extrapolate a missing side.
    a, b = below.iloc[-1], above.iloc[0]
    return a[iv_column] + (b[iv_column] - a[iv_column]) * (1 - a.moneyness_strike_to_spot) / (b.moneyness_strike_to_spot - a.moneyness_strike_to_spot)


def main():
    data = pd.read_csv(ROOT / 'data/processed/options_bsm_ready.csv')
    results = [solve_iv(row.spot_price, row.strike_price, row.time_to_expiry_years,
                        row.risk_free_rate, row.dividend_yield, row.market_price,
                        row.option_type) for row in data.itertuples()]
    data[['iv', 'solver_status', 'iterations', 'price_error']] = pd.DataFrame(results, index=data.index)
    data['py_vollib_iv_check'] = [
        reference_iv(
            row.spot_price, row.strike_price, row.time_to_expiry_years,
            row.risk_free_rate, row.dividend_yield, row.market_price,
            row.option_type,
        )
        for row in data.itertuples()
    ]
    data['iv_check_difference'] = data.iv - data.py_vollib_iv_check
    data['iv_check_passed'] = data.iv_check_difference.abs() <= REFERENCE_IV_TOL
    records = []
    for date, group in data.groupby('valuation_date'):
        call, put = [atm(group.loc[group.option_type.eq(kind)]) for kind in ['CE', 'PE']]
        reference_call, reference_put = [
            atm(group.loc[group.option_type.eq(kind)], 'py_vollib_iv_check')
            for kind in ['CE', 'PE']
        ]
        records.append({'valuation_date': date, 'call_atm_iv': call, 'put_atm_iv': put,
                        'atm_iv': (call + put) / 2, 'contracts': len(group),
                        'failed_solutions': int(group.iv.isna().sum()),
                        'py_vollib_call_atm_iv': reference_call,
                        'py_vollib_put_atm_iv': reference_put,
                        'py_vollib_atm_iv': (reference_call + reference_put) / 2})
    summary = pd.DataFrame(records)
    parity = data.pivot_table(
        index=['valuation_date', 'expiry_date', 'strike_price', 'spot_price',
               'time_to_expiry_years', 'risk_free_rate', 'dividend_yield'],
        columns='option_type', values='market_price'
    ).dropna().reset_index()
    parity['parity_error'] = parity['CE'] - parity['PE'] - (
        parity.spot_price * np.exp(-parity.dividend_yield * parity.time_to_expiry_years)
        - parity.strike_price * np.exp(-parity.risk_free_rate * parity.time_to_expiry_years)
    )
    parity_stats = parity.groupby('valuation_date', as_index=False).agg(
        paired_strikes=('strike_price', 'size'),
        mean_parity_error=('parity_error', 'mean'),
        max_abs_parity_error=('parity_error', lambda x: x.abs().max()),
    )
    summary['atm_call_put_gap'] = (summary.call_atm_iv - summary.put_atm_iv).abs()
    summary['atm_iv_check_difference'] = (
        summary.atm_iv - summary.py_vollib_atm_iv
    )
    summary['contracts_volume_below_10'] = summary.valuation_date.map(
        data.loc[data.traded_volume < 10].groupby('valuation_date').size()
    ).fillna(0).astype(int)
    summary = summary.merge(parity_stats, on='valuation_date', how='left')
    summary['atm_gap_over_10pp'] = summary['atm_call_put_gap'] > 0.10

    assert len(data) == 302, 'Unexpected BSM-ready contract count'
    assert data.valuation_date.nunique() == 12, 'Expected all 12 valuation dates'
    assert not data.duplicated(['valuation_date', 'instrument_id']).any(), 'Duplicate contracts found'
    assert data.iv.notna().all(), 'One or more IV solutions failed'
    assert data.price_error.abs().max() < TOL, 'Repricing tolerance failed'
    assert data.py_vollib_iv_check.notna().all(), 'One or more py_vollib checks failed'
    assert data.iv_check_passed.all(), 'Own IV differs unexpectedly from py_vollib'
    assert summary[['call_atm_iv', 'put_atm_iv', 'atm_iv']].notna().all().all(), 'Missing ATM bracket'
    assert summary.atm_iv_check_difference.abs().max() <= REFERENCE_IV_TOL, 'ATM IV check failed'
    out = ROOT / 'data/processed'
    data.to_csv(out / 'option_implied_volatility.csv', index=False)
    validation_columns = [
        'valuation_date', 'instrument_id', 'option_type', 'expiry_date',
        'strike_price', 'spot_price', 'market_price', 'iv',
        'py_vollib_iv_check', 'iv_check_difference', 'iv_check_passed',
    ]
    data[validation_columns].to_csv(out / 'iv_solver_validation.csv', index=False)
    summary.to_csv(out / 'atm_implied_volatility.csv', index=False)
    parity.to_csv(out / 'option_put_call_parity_diagnostics.csv', index=False)
    print(summary.to_string(index=False))
    print('Solver status:', data.solver_status.value_counts().to_dict())
    print('Maximum absolute repricing error:', data.price_error.abs().max())
    print('Maximum absolute IV difference vs py_vollib:',
          data.iv_check_difference.abs().max())
    print('py_vollib validation:', f'{int(data.iv_check_passed.sum())}/{len(data)} passed')
    print('\nDates requiring ATM data-quality review')
    print(summary.loc[summary.atm_gap_over_10pp, [
        'valuation_date', 'call_atm_iv', 'put_atm_iv', 'atm_call_put_gap',
        'max_abs_parity_error'
    ]].to_string(index=False))
    figures = ROOT / 'outputs/figures'
    figures.mkdir(parents=True, exist_ok=True)
    fig, ax = plt.subplots(figsize=(10, 5.5))
    colors = {'CE': '#1f77b4', 'PE': '#ff7f0e'}
    for kind, label in [('CE', 'Calls'), ('PE', 'Puts')]:
        smile = data.loc[data.valuation_date.eq('2026-09-30') & data.option_type.eq(kind)].sort_values('moneyness_strike_to_spot')
        sizes = 35 + 18 * np.log1p(smile.traded_volume)
        ax.scatter(smile.moneyness_strike_to_spot, smile.iv * 100, s=sizes,
                   alpha=.72, edgecolor='white', linewidth=.7,
                   color=colors[kind], label=f'{label} observations')
        # A liquidity-weighted quadratic shows the broad smile/skew while the
        # scatter points retain every assignment-valid contract.
        weights = np.sqrt(smile.traded_volume)
        coefficients = np.polyfit(
            smile.moneyness_strike_to_spot, smile.iv * 100, 2, w=weights
        )
        curve_x = np.linspace(
            smile.moneyness_strike_to_spot.min(),
            smile.moneyness_strike_to_spot.max(),
            200,
        )
        ax.plot(curve_x, np.polyval(coefficients, curve_x),
                color=colors[kind], linewidth=2.1,
                label=f'{label} weighted curve')
        thin = smile.loc[smile.traded_volume < 10]
        for row in thin.itertuples():
            ax.annotate(f'Vol {int(row.traded_volume)}',
                        (row.moneyness_strike_to_spot, row.iv * 100),
                        xytext=(5, 6), textcoords='offset points', fontsize=8)
    sep = summary.loc[summary.valuation_date.eq('2026-09-30')].iloc[0]
    ax.scatter([1, 1], [sep.call_atm_iv * 100, sep.put_atm_iv * 100],
               marker='X', s=90, color=['#1f77b4', '#ff7f0e'], zorder=4)
    ax.axvline(1, color='#64748b', linewidth=.8, linestyle='--')
    ax.set(xlabel='Moneyness K/S', ylabel='Annualised IV (%)', title='Biocon IV smile_30 September 2026')
    ax.grid(axis='y', alpha=.25)
    ax.legend()
    fig.text(.01, .01, 'Settlement prices. Marker size reflects volume; curves are liquidity-weighted quadratic fits.', fontsize=9, color='#475569')
    fig.tight_layout()
    fig.savefig(figures / 'iv_smile_2026_09_30.png', dpi=180)

    fig, ax = plt.subplots(figsize=(10, 5))
    ax.plot(pd.to_datetime(summary.valuation_date), summary.atm_iv * 100, 'o-')
    ax.set(xlabel='Valuation date', ylabel='Annualised ATM IV (%)', title='Biocon monthly ATM implied volatility')
    fig.autofmt_xdate()
    fig.tight_layout()
    fig.savefig(figures / 'atm_iv_monthly.png', dpi=180)

    # Compare the assignment IV and the repository check on one simple chart.
    comparison_dates = pd.to_datetime(summary.valuation_date)
    fig, ax = plt.subplots(figsize=(10, 5.5))
    ax.plot(
        comparison_dates, summary.py_vollib_atm_iv * 100,
        's--', color='#f59e0b', linewidth=2.2, markersize=6,
        label='py_vollib ATM IV (check only)',
    )
    ax.plot(
        comparison_dates, summary.atm_iv * 100,
        'o-', color='#2563eb', linewidth=1.7, markersize=4,
        label='Own Newton-Raphson ATM IV',
    )
    ax.set(
        xlabel='Valuation date',
        ylabel='Annualised ATM IV (%)',
        title='Biocon ATM IV validation: own solver vs py_vollib',
    )
    ax.grid(axis='y', alpha=.25)
    ax.legend()
    fig.autofmt_xdate()
    fig.tight_layout()
    fig.savefig(figures / 'atm_iv_own_vs_py_vollib.png', dpi=180)
    plt.show()


if __name__ == '__main__':
    main()

In [ ]:
%%writefile src/merton_model.py
"""Part A: Merton DD and PD using ATM IV as equity volatility."""
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from merton.calibration import kmv_iterative
from scipy.optimize import fsolve
from scipy.stats import norm

import input_data

ROOT = Path(__file__).resolve().parents[1]
PD_FLOOR = 0.0003

MARKET_FILE = input_data.file(input_data.MARKET_FILE)
FINANCIAL_FILE = input_data.file(input_data.FINANCIAL_FILE)

# Publication dates are required to preserve the information cutoff.
PUBLICATION_DATES = pd.to_datetime([
    '2025-08-07', '2025-11-11', '2026-02-12',
    '2026-05-07', '2026-08-05',
])


def debt_schedule():
    raw = pd.read_excel(
        FINANCIAL_FILE, sheet_name='Quarterly balance sheet ', header=None
    )
    headers = raw.iloc[2, 1:6].astype(str)
    borrowings = raw.loc[
        raw.iloc[:, 0].astype(str).str.strip().eq('Borrowings'), 1:5
    ].apply(pd.to_numeric)
    assert borrowings.shape == (2, 5)
    reports = headers.str.split('\n').str[0].to_list()
    balance_dates = pd.to_datetime(
        headers.str.split('\n').str[-1].str.replace(',', '', regex=False)
    )
    debt = pd.DataFrame({
        'report': reports,
        'balance_sheet_date': balance_dates,
        'publication_date': PUBLICATION_DATES,
        'non_current_borrowings': borrowings.iloc[0].to_numpy(),
        'current_borrowings': borrowings.iloc[1].to_numpy(),
    })
    debt['debt'] = (
        debt.current_borrowings + 0.5 * debt.non_current_borrowings
    )
    return debt


def share_schedule():
    shares = pd.read_excel(MARKET_FILE, sheet_name='Shares')
    shares = shares.rename(columns={'effective_date': 'share_date'})
    shares['share_date'] = pd.to_datetime(shares.share_date)
    return shares[['share_date', 'shares_outstanding']].sort_values('share_date')


def equations(x, equity, debt, rate, sigma_e):
    """The two scaled Merton equations."""
    value, sigma_v = x
    if value <= 0 or sigma_v <= 0:
        return [1e6, 1e6]
    d1 = (np.log(value / debt) + rate + 0.5 * sigma_v**2) / sigma_v
    d2 = d1 - sigma_v
    model_equity = value * norm.cdf(d1) - debt * np.exp(-rate) * norm.cdf(d2)
    model_equity_vol = norm.cdf(d1) * sigma_v * value
    return [
        (model_equity - equity) / equity,
        (model_equity_vol - sigma_e * equity) / (sigma_e * equity),
    ]


def solve(equity, debt, rate, sigma_e):
    """Authoritative assignment solver using scipy.optimize.fsolve."""
    start = [equity + debt, sigma_e * equity / (equity + debt)]
    args = (equity, debt, rate, sigma_e)
    solution, _, _, _ = fsolve(
        equations, start, args=args, xtol=1e-11, full_output=True
    )
    residual = np.max(np.abs(equations(solution, *args)))
    success = min(solution) > 0 and residual < 1e-8
    return solution[0], solution[1], success, residual


def repository_check(equity, debt, rate, sigma_e):
    """Check-only calibration using the assignment-listed ItsSypher/merton."""
    fit = kmv_iterative(
        equity=equity, equity_vol=sigma_e, debt=debt, rf=rate, T=1,
        tol=1e-10, max_iter=500,
    )
    check_dd = dd(fit.asset_value, debt, fit.asset_vol, rate)
    check_pd = float(fit.diagnostics['risk_neutral_edf'])
    return fit.asset_value, fit.asset_vol, check_dd, check_pd, fit.converged


def dd(value, debt, sigma_v, drift):
    return (np.log(value / debt) + drift - 0.5 * sigma_v**2) / sigma_v


def inputs(volatility=None):
    """Build dated Merton inputs for ATM IV or another equity-volatility series."""
    if volatility is None:
        volatility = pd.read_csv(
            ROOT / 'data/processed/atm_implied_volatility.csv'
        )[['valuation_date', 'atm_iv']]
        volatility['sigma_e'] = volatility.atm_iv
    else:
        volatility = volatility[['valuation_date', 'sigma_e']].copy()
    volatility['valuation_date'] = pd.to_datetime(volatility.valuation_date)
    price = pd.read_csv(ROOT / 'data/processed/valuation_date_equity_prices.csv')
    price['valuation_date'] = pd.to_datetime(price.target_date)
    rate = pd.read_csv(ROOT / 'data/processed/valuation_date_tbill_rates.csv')
    rate = rate.loc[rate.maturity_days.eq(364)].copy()
    rate['valuation_date'] = pd.to_datetime(rate.target_date)

    data = volatility.merge(
        price[['valuation_date', 'close_price']], on='valuation_date'
    ).merge(
        rate[['valuation_date', 'date', 'yield_percent', 'continuous_rate']]
        .rename(columns={'date': 'rate_date'}), on='valuation_date'
    ).sort_values('valuation_date')

    shares = share_schedule()
    data = pd.merge_asof(data, shares, left_on='valuation_date',
                         right_on='share_date', direction='backward')

    debt = debt_schedule()
    data = pd.merge_asof(data, debt.sort_values('publication_date'),
                         left_on='valuation_date', right_on='publication_date',
                         direction='backward')

    history = pd.read_csv(ROOT / 'data/processed/biocon_equity_adjusted.csv')
    history['date'] = pd.to_datetime(history.date)
    data['physical_drift'] = [
        history.loc[history.date <= date, 'model_return'].dropna().mean() * 252
        for date in data.valuation_date
    ]
    data['drift_observations'] = [
        history.loc[history.date <= date, 'model_return'].notna().sum()
        for date in data.valuation_date
    ]
    data['equity_value'] = data.close_price * data.shares_outstanding / 1e6
    data['risk_free_rate'] = data.continuous_rate
    return data.reset_index(drop=True)


def calculate(data):
    rows = []
    for row in data.itertuples():
        main = solve(row.equity_value, row.debt, row.risk_free_rate, row.sigma_e)
        check = repository_check(
            row.equity_value, row.debt, row.risk_free_rate, row.sigma_e
        )
        rows.append({
            'asset_value': main[0], 'sigma_v': main[1],
            'solver_converged': main[2], 'max_scaled_residual': main[3],
            'check_asset_value': check[0], 'check_sigma_v': check[1],
            'check_dd_risk_neutral': check[2],
            'check_pd_risk_neutral_raw': check[3],
            'checker_converged': check[4],
        })
    result = pd.concat([data, pd.DataFrame(rows)], axis=1)
    result['asset_value_check_difference'] = result.asset_value - result.check_asset_value
    result['sigma_v_check_difference'] = result.sigma_v - result.check_sigma_v
    result['debt_to_asset'] = result.debt / result.asset_value
    result['dd_risk_neutral'] = dd(
        result.asset_value, result.debt, result.sigma_v, result.risk_free_rate
    )
    result['dd_check_difference'] = (
        result.dd_risk_neutral - result.check_dd_risk_neutral
    )
    result['dd_physical'] = dd(
        result.asset_value, result.debt, result.sigma_v, result.physical_drift
    )
    for case in ['risk_neutral', 'physical']:
        result[f'pd_{case}_raw'] = norm.cdf(-result[f'dd_{case}'])
        result[f'pd_{case}'] = result[f'pd_{case}_raw'].clip(lower=PD_FLOOR)
        result[f'pd_{case}_bp'] = result[f'pd_{case}'] * 10000
    result['pd_check_difference'] = (
        result.pd_risk_neutral_raw - result.check_pd_risk_neutral_raw
    )
    return result


def sensitivity(main):
    rows = []
    for sigma_e in np.arange(0.10, 0.801, 0.01):
        value, sigma_v, success, residual = solve(
            main.equity_value, main.debt, main.risk_free_rate, sigma_e
        )
        rows.append({
            'sigma_e': sigma_e, 'asset_value': value, 'sigma_v': sigma_v,
            'dd_risk_neutral': dd(value, main.debt, sigma_v,
                                  main.risk_free_rate),
            'solver_converged': success, 'max_scaled_residual': residual,
        })
    return pd.DataFrame(rows)


def figures(result, curve):
    folder = ROOT / 'outputs/figures'
    folder.mkdir(parents=True, exist_ok=True)
    dates = pd.to_datetime(result.valuation_date)

    fig, ax = plt.subplots(figsize=(10, 5.5))
    ax.plot(dates, result.dd_risk_neutral, 'o-', label='Risk-neutral DD')
    ax.plot(dates, result.dd_physical, 's--', label='Physical DD')
    ax.set(xlabel='Valuation date', ylabel='Distance to default',
           title='Biocon Merton distance to default')
    ax.grid(axis='y', alpha=.25)
    ax.legend()
    fig.autofmt_xdate()
    fig.tight_layout()
    fig.savefig(folder / 'merton_dd_over_time.png', dpi=180)

    main = result.iloc[-1]
    fig, ax = plt.subplots(figsize=(9, 5.5))
    ax.plot(curve.sigma_e * 100, curve.dd_risk_neutral, linewidth=2.2)
    ax.scatter(main.sigma_e * 100, main.dd_risk_neutral, marker='X', s=90,
               color='#111827', label='30 Sep 2026 ATM IV')
    ax.set(xlabel='Equity volatility σE (%)', ylabel='Risk-neutral DD',
           title='Biocon Merton DD sensitivity_30 September 2026')
    ax.grid(axis='y', alpha=.25)
    ax.legend()
    fig.tight_layout()
    fig.savefig(folder / 'merton_dd_vs_sigma_e_2026_09_30.png', dpi=180)
    plt.show()


def main():
    model_inputs = inputs()
    result = calculate(model_inputs)
    curve = sensitivity(result.iloc[-1])

    assert len(result) == 12 and model_inputs.notna().all().all()
    assert (result.publication_date <= result.valuation_date).all()
    assert result.solver_converged.all() and result.checker_converged.all()
    assert result.max_scaled_residual.max() < 1e-8
    assert result.asset_value_check_difference.abs().max() < 0.001
    assert result.sigma_v_check_difference.abs().max() < 1e-6
    assert result.dd_check_difference.abs().max() < 1e-5
    assert result.pd_check_difference.abs().max() < 1e-10
    assert curve.solver_converged.all()

    out = ROOT / 'data/processed'
    model_inputs.to_csv(out / 'merton_inputs.csv', index=False)
    result.to_csv(out / 'merton_results.csv', index=False)
    result[[
        'valuation_date', 'asset_value', 'check_asset_value',
        'asset_value_check_difference', 'sigma_v', 'check_sigma_v',
        'sigma_v_check_difference', 'dd_risk_neutral',
        'check_dd_risk_neutral', 'dd_check_difference',
        'pd_risk_neutral_raw', 'check_pd_risk_neutral_raw',
        'pd_check_difference', 'solver_converged', 'checker_converged',
    ]].to_csv(out / 'merton_solver_validation.csv', index=False)
    curve.to_csv(out / 'merton_sigma_e_sensitivity.csv', index=False)

    columns = [
        'valuation_date', 'equity_value', 'debt', 'sigma_e', 'asset_value',
        'sigma_v', 'dd_risk_neutral', 'pd_risk_neutral_bp',
        'dd_physical', 'pd_physical_bp',
    ]
    print(result[columns].to_string(index=False))

    # Make the assignment's low-leverage interpretation explicit.
    main_result = result.iloc[-1]
    floor_dates = int(result.pd_risk_neutral_raw.lt(PD_FLOOR).sum())
    print(
        f'\nInterpretation: 30 Sep 2026 D/V is '
        f'{main_result.debt_to_asset:.1%}. The high DD reflects Biocon\'s '
        'relatively low structural leverage under the Merton default point.'
    )
    print(
        f'PD floor: raw risk-neutral PD is below 3 bp on '
        f'{floor_dates}/{len(result)} dates, so those results are reported at '
        'the assignment floor of 3 bp (0.03%).'
    )
    print('\nItsSypher/merton checker: 12/12 passed')
    print('Maximum V difference:', result.asset_value_check_difference.abs().max())
    print('Maximum sigma_V difference:', result.sigma_v_check_difference.abs().max())
    print('Maximum DD difference:', result.dd_check_difference.abs().max())
    print('Maximum raw PD difference:', result.pd_check_difference.abs().max())
    figures(result, curve)


if __name__ == '__main__':
    main()

In [ ]:
%%writefile src/garch_model.py
"""Part B: rolling GARCH(1,1) volatility and Merton DD/PD."""
from pathlib import Path
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from arch import arch_model
from statsmodels.stats.diagnostic import acorr_ljungbox, het_arch

import merton_model


ROOT = Path(__file__).resolve().parents[1]
DATES = pd.to_datetime([
    '2025-10-31', '2025-11-28', '2025-12-31', '2026-01-30',
    '2026-02-27', '2026-03-30', '2026-04-30', '2026-05-29',
    '2026-06-30', '2026-07-31', '2026-08-31', '2026-09-30',
])
WINDOW, HORIZON = 1000, 252


def annualised_volatility(fit):
    """Assignment formula for expected average variance over 252 days."""
    omega = fit.params['omega'] / 10000
    persistence = fit.params['alpha[1]'] + fit.params['beta[1]']
    long_run = omega / (1 - persistence)
    current = fit.conditional_volatility.iloc[-1] ** 2 / 10000
    speed = np.log(1 / persistence)
    weight = -np.expm1(-speed * HORIZON) / (speed * HORIZON)
    sigma_e = np.sqrt(HORIZON * (long_run + weight * (current - long_run)))
    return sigma_e, persistence, long_run, current


def fit_garch(window, date, distribution):
    returns = window.model_return * 100
    fit = arch_model(
        returns, mean='Constant', vol='GARCH', p=1, q=1,
        dist=distribution, rescale=False,
    ).fit(disp='off', show_warning=False)

    sigma_e, persistence, long_run, current = annualised_volatility(fit)
    direct_variance = fit.forecast(
        horizon=HORIZON, method='analytic', reindex=False
    ).variance.iloc[-1].to_numpy() / 10000
    direct_sigma_e = np.sqrt(HORIZON * direct_variance.mean())

    z = pd.Series(fit.std_resid).dropna()
    lb_z = acorr_ljungbox(z, lags=[10, 20], return_df=True)
    lb_z2 = acorr_ljungbox(z**2, lags=[10, 20], return_df=True)
    with warnings.catch_warnings():
        warnings.filterwarnings('ignore', message='acorr_lm currently returns.*')
        lm_stat, lm_pvalue, _, _ = het_arch(returns - returns.mean(), nlags=10)

    p, se = fit.params, fit.std_err
    return {
        'valuation_date': date,
        'window_start': window.date.iloc[0],
        'window_end': window.date.iloc[-1],
        'observations': len(window),
        'distribution': distribution,
        'converged': fit.convergence_flag == 0,
        'mu': p.get('mu', np.nan) / 100,
        'mu_se': se.get('mu', np.nan) / 100,
        'omega': p['omega'] / 10000,
        'omega_se': se['omega'] / 10000,
        'alpha': p['alpha[1]'], 'alpha_se': se['alpha[1]'],
        'beta': p['beta[1]'], 'beta_se': se['beta[1]'],
        'nu': p.get('nu', np.nan), 'nu_se': se.get('nu', np.nan),
        'persistence': persistence,
        'long_run_daily_variance': long_run,
        'current_daily_variance': current,
        'half_life_days': np.log(0.5) / np.log(persistence),
        'log_likelihood': fit.loglikelihood, 'aic': fit.aic, 'bic': fit.bic,
        'arch_lm_stat_10': lm_stat, 'arch_lm_pvalue_10': lm_pvalue,
        'ljung_box_resid_pvalue_10': lb_z.loc[10, 'lb_pvalue'],
        'ljung_box_resid_pvalue_20': lb_z.loc[20, 'lb_pvalue'],
        'ljung_box_squared_pvalue_10': lb_z2.loc[10, 'lb_pvalue'],
        'ljung_box_squared_pvalue_20': lb_z2.loc[20, 'lb_pvalue'],
        'garch_sigma_e': sigma_e,
        'arch_sigma_e_check': direct_sigma_e,
        'sigma_e_check_difference': sigma_e - direct_sigma_e,
    }


def estimate_models():
    history = pd.read_csv(ROOT / 'data/processed/biocon_equity_adjusted.csv')
    history['date'] = pd.to_datetime(history.date)
    history = history.dropna(subset=['model_return']).sort_values('date')

    rows = []
    for date in DATES:
        window = history.loc[history.date <= date].tail(WINDOW)
        if len(window) != WINDOW:
            raise ValueError(f'{date.date()}: expected {WINDOW} returns')
        rows += [fit_garch(window, date, dist) for dist in ['normal', 't']]
    return pd.DataFrame(rows)


def select_student_t(models):
    """Use Student-t consistently because it fits heavy-tailed returns better."""
    chosen = models.query("distribution == 't'").copy()
    normal = models.query("distribution == 'normal'")[
        ['valuation_date', 'bic', 'garch_sigma_e']
    ].rename(columns={'bic': 'normal_bic', 'garch_sigma_e': 'normal_sigma_e'})
    chosen = chosen.merge(normal, on='valuation_date', validate='one_to_one')
    chosen = chosen.rename(columns={
        'bic': 'student_t_bic', 'garch_sigma_e': 'student_t_sigma_e'
    })
    chosen['student_t_bic_better'] = chosen.student_t_bic < chosen.normal_bic
    chosen['sigma_e'] = chosen.student_t_sigma_e
    return chosen


def plot_results(forecasts, merton):
    folder = ROOT / 'outputs/figures'
    folder.mkdir(parents=True, exist_ok=True)
    dates = pd.to_datetime(forecasts.valuation_date)

    fig, ax = plt.subplots(figsize=(10, 5.5))
    ax.plot(dates, forecasts.normal_sigma_e * 100, 'o--', label='Normal GARCH')
    ax.plot(dates, forecasts.student_t_sigma_e * 100, 's-', label='Student-t GARCH')
    ax.set(xlabel='Valuation date', ylabel='Annualised equity volatility (%)',
           title='Biocon rolling 1,000-day GARCH(1,1) volatility')
    ax.grid(axis='y', alpha=.25)
    ax.legend()
    fig.autofmt_xdate()
    fig.tight_layout()
    fig.savefig(folder / 'garch_sigma_e_over_time.png', dpi=180)

    fig, ax = plt.subplots(figsize=(10, 5.5))
    ax.plot(dates, merton.dd_risk_neutral, 'o-', label='Risk-neutral DD')
    ax.plot(dates, merton.dd_physical, 's--', label='Physical DD')
    ax.set(xlabel='Valuation date', ylabel='Distance to default',
           title='Biocon Merton DD using Student-t GARCH volatility')
    ax.grid(axis='y', alpha=.25)
    ax.legend()
    fig.autofmt_xdate()
    fig.tight_layout()
    fig.savefig(folder / 'garch_merton_dd_over_time.png', dpi=180)
    plt.show()


def main():
    models = estimate_models()
    forecasts = select_student_t(models)
    inputs = merton_model.inputs(forecasts[['valuation_date', 'sigma_e']])
    merton = merton_model.calculate(inputs)

    assert len(models) == 24 and len(forecasts) == 12
    assert models.observations.eq(WINDOW).all()
    assert (models.window_end <= models.valuation_date).all()
    assert models.converged.all()
    assert (models.omega > 0).all()
    assert (models[['alpha', 'beta']] >= 0).all().all()
    assert models.persistence.between(0, 1, inclusive='neither').all()
    assert models.sigma_e_check_difference.abs().max() < 0.001
    assert merton.solver_converged.all() and merton.max_scaled_residual.max() < 1e-8
    assert merton.checker_converged.all()
    assert merton.asset_value_check_difference.abs().max() < 0.001
    assert merton.sigma_v_check_difference.abs().max() < 1e-6
    assert merton.dd_check_difference.abs().max() < 1e-5

    out = ROOT / 'data/processed'
    models.to_csv(out / 'garch_model_comparison.csv', index=False)
    forecasts.to_csv(out / 'garch_volatility_forecasts.csv', index=False)
    merton.to_csv(out / 'garch_merton_results.csv', index=False)

    columns = ['valuation_date', 'window_start', 'window_end', 'persistence',
               'half_life_days', 'normal_sigma_e', 'student_t_sigma_e']
    print('\nGARCH VOLATILITY FORECASTS')
    print(forecasts[columns].to_string(index=False))
    print(f"\nStudent-t BIC wins: {forecasts.student_t_bic_better.sum()}/12")
    print('Maximum volatility check difference:',
          models.sigma_e_check_difference.abs().max())

    selected = models.query("distribution == 't'")
    print('Diagnostics below 5%: '
          f"ARCH-LM {(selected.arch_lm_pvalue_10 < .05).sum()}/12; "
          f"residual LB10 {(selected.ljung_box_resid_pvalue_10 < .05).sum()}/12; "
          f"squared LB10 {(selected.ljung_box_squared_pvalue_10 < .05).sum()}/12")

    result = merton.iloc[-1]
    print('\n30 SEPTEMBER 2026 - GARCH MERTON RESULT')
    print(result[['equity_value', 'debt', 'debt_to_asset', 'sigma_e', 'sigma_v',
                  'dd_risk_neutral', 'pd_risk_neutral_bp',
                  'dd_physical', 'pd_physical_bp']].to_string())
    plot_results(forecasts, merton)


if __name__ == '__main__':
    main()

In [ ]:
%%writefile src/ml_volatility.py
"""Part B: Random Forest volatility forecasts and Merton DD/PD."""
from pathlib import Path
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from arch import arch_model
from sklearn.ensemble import RandomForestRegressor

import merton_model
import input_data


ROOT = Path(__file__).resolve().parents[1]
MARKET_FILE = input_data.file(input_data.MARKET_FILE)
DATES = pd.to_datetime([
    '2025-10-31', '2025-11-28', '2025-12-31', '2026-01-30',
    '2026-02-27', '2026-03-30', '2026-04-30', '2026-05-29',
    '2026-06-30', '2026-07-31', '2026-08-31', '2026-09-30',
])
FEATURES = [
    'rv_5', 'rv_21', 'rv_63', 'return_1', 'return_5', 'return_21',
    'garch_sigma_t', 'india_vix', 'nifty_return_21', 'results_month',
]
WINDOW, MIN_GARCH = 1000, 252


def daily_garch(history):
    """Create known-at-date GARCH features and 21/252-day benchmarks."""
    cache = ROOT / 'data/processed/ml_daily_garch_features.csv'
    expected = len(history) - MIN_GARCH + 1
    if cache.exists():
        saved = pd.read_csv(cache, parse_dates=['date'])
        if (len(saved) == expected and saved.date.iloc[-1] == history.date.iloc[-1]):
            return saved

    rows = []
    returns = history.model_return.to_numpy()
    for i in range(MIN_GARCH - 1, len(history)):
        window = returns[max(0, i - WINDOW + 1):i + 1] * 100
        with warnings.catch_warnings():
            warnings.simplefilter('ignore')
            fit = arch_model(
                window, mean='Constant', vol='GARCH', p=1, q=1,
                dist='t', rescale=False,
            ).fit(disp='off', show_warning=False, options={'maxiter': 500})
        variance = fit.forecast(
            horizon=252, method='analytic', reindex=False
        ).variance.iloc[-1].to_numpy() / 10000
        rows.append({
            'date': history.date.iloc[i],
            'garch_sigma_t': fit.conditional_volatility[-1] / 100 * np.sqrt(252),
            'garch_sigma_21': np.sqrt(252 * variance[:21].mean()),
            'garch_sigma_252': np.sqrt(252 * variance.mean()),
            'garch_converged': fit.convergence_flag == 0,
        })
        if len(rows) % 500 == 0:
            print(f'Daily GARCH features: {len(rows)}/{expected}')

    result = pd.DataFrame(rows)
    result.to_csv(cache, index=False)
    return result


def build_panel():
    """Combine only information known by each Biocon trading date."""
    equity = pd.read_csv(ROOT / 'data/processed/biocon_equity_adjusted.csv')
    equity['date'] = pd.to_datetime(equity.date)
    equity = equity.dropna(subset=['model_return']).sort_values('date')

    vix = pd.read_excel(MARKET_FILE, sheet_name='India-VIX')
    vix.columns = vix.columns.str.strip()
    vix = vix.rename(columns={'Date': 'date', 'Close': 'india_vix'})
    vix['date'] = pd.to_datetime(vix.date)

    nifty = pd.read_excel(MARKET_FILE, sheet_name='Nifty-50')
    nifty.columns = nifty.columns.str.strip()
    nifty = nifty.rename(columns={'Date': 'date', 'Close': 'nifty_close'})
    nifty['date'] = pd.to_datetime(nifty.date)
    nifty = nifty.sort_values('date')
    nifty['nifty_return_21'] = np.log(
        nifty.nifty_close / nifty.nifty_close.shift(21)
    )

    panel = equity[['date', 'model_return']].merge(
        nifty[['date', 'nifty_close', 'nifty_return_21']],
        on='date', how='left', validate='one_to_one',
    ).merge(
        vix[['date', 'india_vix']], on='date', how='left', validate='one_to_one'
    )
    missing_vix = panel.loc[panel.india_vix.isna(), 'date']
    assert missing_vix.tolist() == [pd.Timestamp('2024-03-02')]
    panel['india_vix'] = panel.india_vix.ffill()

    result_dates = pd.read_excel(
        MARKET_FILE, sheet_name='Biocon_quartely-results-date'
    )['result_month'].astype(str)
    panel['results_month'] = (
        panel.date.dt.to_period('M').astype(str).isin(set(result_dates)).astype(int)
    )
    panel['return_1'] = panel.model_return
    for days in [5, 21]:
        panel[f'return_{days}'] = panel.model_return.rolling(days).sum()
    for days in [5, 21, 63]:
        panel[f'rv_{days}'] = np.sqrt(
            252 * panel.model_return.pow(2).rolling(days).mean()
        )

    panel = panel.merge(daily_garch(equity), on='date', how='left')
    for horizon in [21, 252]:
        future = panel.model_return.shift(-1)
        panel[f'target_sigma_{horizon}'] = np.sqrt(
            252 * future.pow(2).iloc[::-1].rolling(horizon).mean().iloc[::-1]
        )
        panel[f'target_end_{horizon}'] = panel.date.shift(-horizon)
    return panel


def metrics(actual, forecast, benchmark):
    """Assignment accuracy measures on annualised volatility."""
    actual, forecast, benchmark = map(
        np.asarray, (actual, forecast, benchmark)
    )
    ratio = actual**2 / np.maximum(forecast**2, 1e-12)
    return {
        'rmse': np.sqrt(np.mean((actual - forecast) ** 2)),
        'mae': np.mean(np.abs(actual - forecast)),
        'qlike': np.mean(ratio - np.log(ratio) - 1),
        'oos_r2': 1 - np.mean((actual - forecast) ** 2)
        / np.mean((actual - benchmark) ** 2),
    }


def time_split(panel, horizon):
    """Use 60/20/20 chronological splits and purge overlapping labels."""
    target = f'target_sigma_{horizon}'
    end = f'target_end_{horizon}'
    data = panel.dropna(subset=FEATURES + [target, end, f'garch_sigma_{horizon}'])
    data = data.loc[data[end] < DATES[0]].reset_index(drop=True)
    validation_start = data.date.iloc[int(len(data) * .60)]
    test_start = data.date.iloc[int(len(data) * .80)]
    train = data.loc[(data.date < validation_start) & (data[end] < validation_start)]
    validation = data.loc[
        (data.date >= validation_start) & (data.date < test_start)
        & (data[end] < test_start)
    ]
    test = data.loc[data.date >= test_start]
    return train, validation, test, validation_start, test_start


def new_model(params):
    return RandomForestRegressor(
        random_state=42, n_jobs=-1, max_features=.8, **params,
    )


def recency_weights(rows, half_life):
    """Give recent rows more weight; None keeps equal weights."""
    if half_life is None:
        return None
    age = len(rows) - 1 - np.arange(len(rows))
    return 0.5 ** (age / half_life)


def select_model(panel, horizon):
    """Tune on validation data, then evaluate once on untouched test data."""
    train, validation, test, validation_start, test_start = time_split(
        panel, horizon
    )
    target = f'target_sigma_{horizon}'
    grid = [
        {'n_estimators': trees, 'max_depth': depth, 'min_samples_leaf': leaf}
        for trees in [300, 600] for depth in [3, 5, None]
        for leaf in [5, 10, 20]
    ]
    tuning = []
    for params in grid:
        model = new_model(params)
        model.fit(train[FEATURES], np.log(train[target]))
        prediction = np.exp(model.predict(validation[FEATURES]))
        score = metrics(
            validation[target], prediction,
            validation[f'garch_sigma_{horizon}'],
        )
        tuning.append({**params, **score})
    tuning = pd.DataFrame(tuning).sort_values(['qlike', 'rmse'])
    winner = tuning.iloc[0]
    best = {
        'n_estimators': int(winner.n_estimators),
        'max_depth': None if pd.isna(winner.max_depth) else int(winner.max_depth),
        'min_samples_leaf': int(winner.min_samples_leaf),
    }

    weight_tuning = []
    for half_life in [None, 252, 504, 756, 1260]:
        model = new_model(best)
        model.fit(
            train[FEATURES], np.log(train[target]),
            sample_weight=recency_weights(train, half_life),
        )
        prediction = np.exp(model.predict(validation[FEATURES]))
        score = metrics(
            validation[target], prediction,
            validation[f'garch_sigma_{horizon}'],
        )
        weight_tuning.append({
            **best, 'weight_half_life': half_life or 0, **score,
        })
    weight_tuning = pd.DataFrame(weight_tuning).sort_values(['qlike', 'rmse'])
    # Recency weighting is a reported sensitivity, not a post-test model change.
    # The original equal-weight specification remains the locked final model.
    selected_half_life = None
    tuning['weight_half_life'] = np.nan
    tuning = pd.concat([tuning, weight_tuning], ignore_index=True)

    development = pd.concat([train, validation]).sort_values('date')
    model = new_model(best).fit(
        development[FEATURES], np.log(development[target]),
        sample_weight=recency_weights(development, selected_half_life),
    )
    ml_prediction = np.exp(model.predict(test[FEATURES]))
    accuracy = []
    for name, prediction in [
        ('Random Forest', ml_prediction),
        ('GARCH(1,1)', test[f'garch_sigma_{horizon}']),
    ]:
        accuracy.append({
            'horizon_days': horizon, 'model': name,
            'train_rows': len(train), 'validation_rows': len(validation),
            'test_rows': len(test), 'validation_start': validation_start,
            'test_start': test_start, 'test_end': test.target_end_252.max()
            if horizon == 252 else test.target_end_21.max(),
            **metrics(test[target], prediction, test[f'garch_sigma_{horizon}']),
        })
    return best, selected_half_life, tuning, pd.DataFrame(accuracy), test


def monthly_forecasts(panel, horizon, params, half_life):
    """Refit with labels observable by each assignment valuation date."""
    target, end = f'target_sigma_{horizon}', f'target_end_{horizon}'
    rows, final_model = [], None
    for date in DATES:
        training = panel.dropna(subset=FEATURES + [target, end])
        training = training.loc[training[end] <= date]
        current = panel.loc[panel.date.eq(date)]
        assert len(current) == 1 and current[FEATURES].notna().all(axis=None)
        final_model = new_model(params).fit(
            training[FEATURES], np.log(training[target]),
            sample_weight=recency_weights(training, half_life),
        )
        rows.append({
            'valuation_date': date, 'horizon_days': horizon,
            'training_rows': len(training),
            'latest_observable_target_end': training[end].max(),
            'ml_sigma_e': np.exp(final_model.predict(current[FEATURES]))[0],
            'garch_sigma_e': current[f'garch_sigma_{horizon}'].iloc[0],
        })
    importance = pd.DataFrame({
        'feature': FEATURES, 'feature_importance': final_model.feature_importances_
    }).sort_values('feature_importance', ascending=False)
    return pd.DataFrame(rows), importance


def figures(forecasts, importance, ml_merton):
    folder = ROOT / 'outputs/figures'
    folder.mkdir(parents=True, exist_ok=True)
    main = forecasts.query('horizon_days == 252')

    fig, ax = plt.subplots(figsize=(10, 5.5))
    ax.plot(main.valuation_date, main.ml_sigma_e * 100, 'o-', label='Random Forest')
    ax.plot(main.valuation_date, main.garch_sigma_e * 100, 's--', label='GARCH')
    ax.set(xlabel='Valuation date', ylabel='Annualised equity volatility (%)',
           title='Biocon 252-day volatility forecasts')
    ax.grid(axis='y', alpha=.25)
    ax.legend()
    fig.autofmt_xdate()
    fig.tight_layout()
    fig.savefig(folder / 'ml_vs_garch_sigma_e_over_time.png', dpi=180)

    fig, ax = plt.subplots(figsize=(9, 5.5))
    ordered = importance.sort_values('feature_importance')
    ax.barh(ordered.feature, ordered.feature_importance, color='#2563eb')
    ax.set(xlabel='Random Forest feature importance',
           title='252-day model feature importance')
    ax.grid(axis='x', alpha=.2)
    fig.tight_layout()
    fig.savefig(folder / 'ml_feature_importance_252.png', dpi=180)

    garch = pd.read_csv(ROOT / 'data/processed/garch_merton_results.csv')
    garch['valuation_date'] = pd.to_datetime(garch.valuation_date)
    fig, ax = plt.subplots(figsize=(10, 5.5))
    ax.plot(ml_merton.valuation_date, ml_merton.dd_risk_neutral,
            'o-', label='Random Forest DD')
    ax.plot(garch.valuation_date, garch.dd_risk_neutral,
            's--', label='GARCH DD')
    ax.set(xlabel='Valuation date', ylabel='Risk-neutral distance to default',
           title='Biocon Merton DD: Random Forest versus GARCH')
    ax.grid(axis='y', alpha=.25)
    ax.legend()
    fig.autofmt_xdate()
    fig.tight_layout()
    fig.savefig(folder / 'ml_vs_garch_merton_dd.png', dpi=180)
    plt.show()


def main():
    panel = build_panel()
    all_forecasts, all_accuracy, all_tuning, importances = [], [], [], {}
    for horizon in [21, 252]:
        params, half_life, tuning, accuracy, _ = select_model(panel, horizon)
        forecasts, importance = monthly_forecasts(
            panel, horizon, params, half_life
        )
        forecasts['weight_half_life'] = half_life or 0
        tuning.insert(0, 'horizon_days', horizon)
        importance.insert(0, 'horizon_days', horizon)
        all_forecasts.append(forecasts)
        all_accuracy.append(accuracy)
        all_tuning.append(tuning)
        importances[horizon] = importance
        print(f'{horizon}-day selected parameters: {params}; '
              f'weight half-life: {half_life or "equal"}')

    forecasts = pd.concat(all_forecasts, ignore_index=True)
    accuracy = pd.concat(all_accuracy, ignore_index=True)
    tuning = pd.concat(all_tuning, ignore_index=True)
    importance = pd.concat(importances.values(), ignore_index=True)

    selected = forecasts.query('horizon_days == 252').rename(
        columns={'ml_sigma_e': 'sigma_e'}
    )[['valuation_date', 'sigma_e']]
    ml_merton = merton_model.calculate(merton_model.inputs(selected))

    assert panel.garch_converged.dropna().all()
    assert len(forecasts) == 24 and len(accuracy) == 4
    assert forecasts[['ml_sigma_e', 'garch_sigma_e']].gt(0).all().all()
    assert (forecasts.latest_observable_target_end <= forecasts.valuation_date).all()
    assert ml_merton.solver_converged.all() and ml_merton.checker_converged.all()
    assert ml_merton.max_scaled_residual.max() < 1e-8
    assert ml_merton.dd_check_difference.abs().max() < 1e-5

    out = ROOT / 'data/processed'
    panel.to_csv(out / 'ml_feature_target_panel.csv', index=False)
    accuracy.to_csv(out / 'ml_test_accuracy.csv', index=False)
    tuning.to_csv(out / 'ml_validation_tuning.csv', index=False)
    forecasts.to_csv(out / 'ml_volatility_forecasts.csv', index=False)
    importance.to_csv(out / 'ml_feature_importance.csv', index=False)
    ml_merton.to_csv(out / 'ml_merton_results.csv', index=False)

    print('\nOUT-OF-SAMPLE ACCURACY')
    print(accuracy[['horizon_days', 'model', 'test_rows', 'rmse', 'mae',
                    'qlike', 'oos_r2']].to_string(index=False))
    print('\n252-DAY MONTHLY FORECASTS')
    print(forecasts.query('horizon_days == 252')[
        ['valuation_date', 'ml_sigma_e', 'garch_sigma_e']
    ].to_string(index=False))
    print('\n30 SEPTEMBER 2026 - RANDOM FOREST MERTON RESULT')
    print(ml_merton.iloc[-1][[
        'sigma_e', 'sigma_v', 'dd_risk_neutral', 'pd_risk_neutral_bp',
        'dd_physical', 'pd_physical_bp',
    ]].to_string())
    figures(forecasts, importances[252], ml_merton)


if __name__ == '__main__':
    main()

In [ ]:
%%writefile src/rating_model.py
"""Part C: CRISIL rating PD and reverse-engineered Merton volatility."""
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.optimize import fsolve, least_squares
from scipy.stats import norm

import merton_model
import input_data


ROOT = Path(__file__).resolve().parents[1]
PD_FLOOR = 0.0003
FINANCIAL_FILE = input_data.file(input_data.FINANCIAL_FILE)


def rating_inputs():
    source = pd.read_excel(
        FINANCIAL_FILE, sheet_name='Rating Data', header=3, nrows=1
    ).iloc[0]
    defaults = pd.read_excel(
        FINANCIAL_FILE, sheet_name='Rating Data', header=7, nrows=3,
        usecols='A:C',
    ).set_index('rating_category')
    main = merton_model.inputs().iloc[-1]

    rating_pd_raw = defaults.loc['AA', 'one_year_default_rate']
    better_pd_raw = defaults.loc['AAA', 'one_year_default_rate']
    worse_pd_raw = defaults.loc['A', 'one_year_default_rate']
    rating_pd = max(rating_pd_raw, PD_FLOOR)
    better_pd = max(better_pd_raw, PD_FLOOR)
    worse_pd = max(worse_pd_raw, PD_FLOOR)

    cases = pd.DataFrame({
        'case': ['lower_edge', 'rating_point', 'upper_edge'],
        'target_pd': [
            np.sqrt(better_pd * rating_pd),
            rating_pd,
            np.sqrt(rating_pd * worse_pd),
        ],
    })
    cases['target_pd_bp'] = cases.target_pd * 10000
    cases['target_dd'] = -norm.ppf(cases.target_pd)
    cases['company'] = 'Biocon Limited'
    cases['valuation_date'] = source['valuation_date']
    cases['agency'] = source['agency']
    cases['long_term_rating'] = source['long_term_rating']
    cases['outlook'] = source['outlook']
    cases['rationale_date'] = source['rationale_date']
    cases['published_pd_category'] = source['published_pd_category']
    cases['published_pd_percent'] = rating_pd_raw * 100
    cases['better_category'] = 'AAA'
    cases['better_pd_percent'] = better_pd_raw * 100
    cases['worse_category'] = 'A'
    cases['worse_pd_percent'] = worse_pd_raw * 100
    cases['rating_pd_raw'] = rating_pd_raw
    cases['rating_pd_after_floor'] = rating_pd
    cases['better_pd_after_floor'] = better_pd
    cases['worse_pd_after_floor'] = worse_pd
    cases['equity_value'] = main.equity_value
    cases['debt'] = main.debt
    cases['risk_free_rate'] = main.risk_free_rate
    cases['physical_drift'] = main.physical_drift
    return cases


def equations(x, equity, debt, rate, drift, target_dd):
    value, sigma_v = x
    if value <= 0 or sigma_v <= 0:
        return [1e6, 1e6]
    d1 = (np.log(value / debt) + rate + 0.5 * sigma_v**2) / sigma_v
    d2 = d1 - sigma_v
    model_equity = value * norm.cdf(d1) - debt * np.exp(-rate) * norm.cdf(d2)
    model_dd = (
        np.log(value / debt) + drift - 0.5 * sigma_v**2
    ) / sigma_v
    return [(model_equity - equity) / equity, model_dd - target_dd]


def solve_case(row, drift_name, drift):
    args = (row.equity_value, row.debt, row.risk_free_rate,
            drift, row.target_dd)
    start = [row.equity_value + row.debt, 0.50]
    solution, _, ier, message = fsolve(
        equations, start, args=args, full_output=True, xtol=1e-11
    )
    residual = np.max(np.abs(equations(solution, *args)))

    check = least_squares(
        lambda x: equations(x, *args), start,
        bounds=([1e-8, 1e-8], [np.inf, 3]),
        xtol=1e-13, ftol=1e-13, gtol=1e-13,
    )
    value, sigma_v = solution
    d1 = (
        np.log(value / row.debt) + row.risk_free_rate + 0.5 * sigma_v**2
    ) / sigma_v
    sigma_e = norm.cdf(d1) * sigma_v * value / row.equity_value
    return {
        'drift_case': drift_name, 'drift': drift,
        'asset_value': value, 'sigma_v': sigma_v, 'sigma_e': sigma_e,
        'fsolve_ier': ier, 'fsolve_message': message,
        'max_scaled_residual': residual,
        'check_success': check.success,
        'check_asset_value': check.x[0], 'check_sigma_v': check.x[1],
        'asset_value_check_difference': value - check.x[0],
        'sigma_v_check_difference': sigma_v - check.x[1],
    }


def calculate(cases):
    rows = []
    for case in cases.itertuples(index=False):
        for name, drift in [
            ('risk_neutral', case.risk_free_rate),
            ('physical', case.physical_drift),
        ]:
            rows.append({**case._asdict(), **solve_case(case, name, drift)})
    return pd.DataFrame(rows)


def market_comparison(results):
    band = results.query("drift_case == 'risk_neutral'").set_index('case')
    lower = band.loc['lower_edge', 'sigma_e']
    upper = band.loc['upper_edge', 'sigma_e']
    iv = pd.read_csv(ROOT / 'data/processed/atm_implied_volatility.csv').iloc[-1].atm_iv
    garch = pd.read_csv(
        ROOT / 'data/processed/garch_volatility_forecasts.csv'
    ).iloc[-1].sigma_e
    comparison = pd.DataFrame({
        'measure': ['ATM implied volatility', 'Student-t GARCH'],
        'sigma_e': [iv, garch],
    })
    comparison['rating_band_lower'] = lower
    comparison['rating_band_upper'] = upper
    comparison['inside_rating_band'] = comparison.sigma_e.between(lower, upper)
    return comparison


def figure(comparison):
    folder = ROOT / 'outputs/figures'
    folder.mkdir(parents=True, exist_ok=True)
    lower = comparison.rating_band_lower.iloc[0] * 100
    upper = comparison.rating_band_upper.iloc[0] * 100
    fig, ax = plt.subplots(figsize=(8.5, 4.8))
    band_colour = '#dc2626'
    ax.axvspan(lower, upper, color=band_colour, alpha=.24,
               label='CRISIL-consistent volatility band')
    ax.axvline(lower, color=band_colour, linewidth=2)
    ax.axvline(upper, color=band_colour, linewidth=2)
    for row in comparison.itertuples():
        ax.scatter(row.sigma_e * 100, 0, s=90, label=row.measure)
    ax.annotate(
        '63.12% to 65.02% volatility is required\n'
        'for Merton PD to reach the CRISIL-implied\n'
        'range of 3.00 to 4.58 bp',
        xy=((lower + upper) / 2, .05),
        xytext=(47, .52),
        ha='center', va='center', fontsize=9.5, color='#7f1d1d',
        bbox=dict(boxstyle='round,pad=.45', facecolor='#fef2f2',
                  edgecolor=band_colour, linewidth=1.4),
        arrowprops=dict(arrowstyle='-|>', color=band_colour, linewidth=1.6),
    )
    ax.set(xlabel='Annualised equity volatility (%)', yticks=[],
           title='Biocon market volatility versus CRISIL-implied band')
    ax.set_ylim(-.28, .72)
    ax.grid(axis='x', alpha=.2)
    ax.legend()
    fig.tight_layout()
    fig.savefig(folder / 'rating_implied_volatility_band.png', dpi=180)
    plt.show()


def main():
    cases = rating_inputs()
    results = calculate(cases)
    comparison = market_comparison(results)

    assert len(results) == 6
    assert results.fsolve_ier.eq(1).all()
    assert results.max_scaled_residual.max() < 1e-8
    assert results.check_success.all()
    assert results.asset_value_check_difference.abs().max() < 0.001
    assert results.sigma_v_check_difference.abs().max() < 1e-8
    assert cases.rating_pd_after_floor.eq(PD_FLOOR).all()
    assert cases.target_pd.between(PD_FLOOR, 0.001).all()

    out = ROOT / 'data/processed'
    cases.to_csv(out / 'rating_pd_bucket.csv', index=False)
    results.to_csv(out / 'rating_reverse_engineered_volatility.csv', index=False)
    comparison.to_csv(out / 'rating_market_comparison.csv', index=False)

    print('\nCRISIL RATING AND PD BUCKET')
    print(cases[[
        'agency', 'long_term_rating', 'outlook', 'rationale_date',
        'case', 'target_pd_bp', 'target_dd',
    ]].to_string(index=False))
    print('\nRATING-IMPLIED VOLATILITY')
    print(results[[
        'case', 'drift_case', 'target_pd_bp', 'target_dd',
        'asset_value', 'sigma_v', 'sigma_e',
    ]].to_string(index=False))
    print('\nMARKET COMPARISON')
    print(comparison.to_string(index=False))
    figure(comparison)


if __name__ == '__main__':
    main()

In [ ]:
%%writefile src/altman_model.py
"""Part D: Altman scores, rating-equivalent PD and implied volatility."""
from pathlib import Path
from types import SimpleNamespace

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import norm

import merton_model
import rating_model
import input_data


ROOT = Path(__file__).resolve().parents[1]
YEARS = [2022, 2023, 2024, 2025, 2026]
FINANCIAL_FILE = input_data.file(input_data.FINANCIAL_FILE)


def em_rating_map():
    frame = pd.read_excel(FINANCIAL_FILE, sheet_name='Altman Rating Map', header=3)
    frame = frame[[
        'rating_equivalent', 'lower_bound', 'upper_bound'
    ]].dropna(subset=['rating_equivalent'])
    frame.columns = ['rating', 'lower', 'upper']
    frame['lower'] = pd.to_numeric(frame.lower, errors='coerce').fillna(-np.inf)
    frame['upper'] = pd.to_numeric(frame.upper, errors='coerce').fillna(np.inf)
    return frame


def row_values(frame, label):
    """Read FY2022-FY2026 values from an exact statement row."""
    rows = frame[frame.iloc[:, 0].astype(str).str.strip().eq(label)]
    assert len(rows) == 1, f'Expected one row for {label}'
    values = pd.to_numeric(rows.iloc[0, 1:6], errors='raise').to_numpy(float)
    assert np.isfinite(values).all()
    return values


def rating_for_score(score):
    for row in em_rating_map().itertuples(index=False):
        if row.lower <= score < row.upper:
            return row.rating
    raise ValueError(f'No rating interval for EM score {score}')


def build_scores():
    source = FINANCIAL_FILE
    balance = pd.read_excel(source, sheet_name='Balance Sheet', header=None)
    profit = pd.read_excel(source, sheet_name='Profit & Loss', header=None)
    prices = pd.read_csv(
        ROOT / 'data/processed/fiscal_year_end_equity_prices.csv'
    ).set_index('fiscal_year').loc[YEARS]

    current_assets = row_values(balance, 'Total Current Assets')
    current_liabilities = row_values(balance, 'Total Current Liabilities')
    total_assets = row_values(balance, 'TOTAL ASSETS')
    retained_earnings = row_values(balance, 'Other equity')
    book_equity = row_values(balance, 'Total Equity')
    total_liabilities = row_values(balance, 'Total Liabilities')
    sales = row_values(profit, 'Revenue from operations')
    pbt = row_values(profit, 'Profit Before Tax')
    finance_cost = row_values(profit, 'Finance costs')
    ebit = pbt + finance_cost
    shares = pd.read_excel(
        input_data.file(input_data.MARKET_FILE), sheet_name='Shares'
    )
    shares.effective_date = pd.to_datetime(shares.effective_date)
    year_ends = pd.DataFrame({'date': pd.to_datetime([f'{y}-03-31' for y in YEARS])})
    annual_shares = pd.merge_asof(
        year_ends.sort_values('date'), shares.sort_values('effective_date'),
        left_on='date', right_on='effective_date', direction='backward',
    ).shares_outstanding.to_numpy()
    market_equity = prices.close_price.to_numpy() * annual_shares / 1e6

    data = pd.DataFrame({
        'fiscal_year': YEARS,
        'current_assets': current_assets,
        'current_liabilities': current_liabilities,
        'working_capital': current_assets - current_liabilities,
        'retained_earnings_other_equity': retained_earnings,
        'profit_before_tax': pbt,
        'finance_cost': finance_cost,
        'ebit': ebit,
        'sales': sales,
        'total_assets': total_assets,
        'total_liabilities': total_liabilities,
        'book_equity': book_equity,
        'shares_outstanding': annual_shares,
        'fiscal_year_end_date': prices.date.to_numpy(),
        'fiscal_year_end_price': prices.close_price.to_numpy(),
        'market_equity': market_equity,
    })
    data['x1_working_capital'] = data.working_capital / data.total_assets
    data['x2_retained_earnings'] = (
        data.retained_earnings_other_equity / data.total_assets
    )
    data['x3_ebit'] = data.ebit / data.total_assets
    data['x4_market_equity'] = data.market_equity / data.total_liabilities
    data['x4_book_equity'] = data.book_equity / data.total_liabilities
    data['x5_sales'] = data.sales / data.total_assets

    data['z_score'] = (
        1.2 * data.x1_working_capital
        + 1.4 * data.x2_retained_earnings
        + 3.3 * data.x3_ebit
        + 0.6 * data.x4_market_equity
        + data.x5_sales
    )
    data['z_double_prime'] = (
        6.56 * data.x1_working_capital
        + 3.26 * data.x2_retained_earnings
        + 6.72 * data.x3_ebit
        + 1.05 * data.x4_book_equity
    )
    data['em_score'] = data.z_double_prime + 3.25
    data['z_zone'] = pd.cut(
        data.z_score, [-np.inf, 1.81, 2.99, np.inf],
        labels=['Distress', 'Grey', 'Safe'], right=False,
    ).astype(str)
    data['z_double_prime_zone'] = pd.cut(
        data.z_double_prime, [-np.inf, 1.10, 2.60, np.inf],
        labels=['Distress', 'Grey', 'Safe'], right=False,
    ).astype(str)
    data['em_zone'] = pd.cut(
        data.em_score, [-np.inf, 4.35, 5.85, np.inf],
        labels=['Distress', 'Grey', 'Safe'], right=False,
    ).astype(str)
    data['em_rating_equivalent'] = data.em_score.map(rating_for_score)
    return data


def reverse_engineer(scores):
    """Map FY2026 EM rating to CRISIL PD and reverse-solve Merton."""
    final = scores.iloc[-1]
    rating_source = pd.read_excel(
        FINANCIAL_FILE, sheet_name='Rating Data', header=7, nrows=3,
        usecols='A:C',
    ).set_index('rating_category')
    category = final.em_rating_equivalent.rstrip('+-')
    if category not in rating_source.index:
        raise ValueError(f'CRISIL default rate not collected for {category}')
    raw_pd = rating_source.loc[category, 'one_year_default_rate']
    target_pd = max(raw_pd, merton_model.PD_FLOOR)
    main = merton_model.inputs().iloc[-1]
    case = SimpleNamespace(
        equity_value=main.equity_value,
        debt=main.debt,
        risk_free_rate=main.risk_free_rate,
        physical_drift=main.physical_drift,
        target_dd=-norm.ppf(target_pd),
    )
    rows = []
    for drift_case, drift in [
        ('risk_neutral', case.risk_free_rate),
        ('physical', case.physical_drift),
    ]:
        solved = rating_model.solve_case(case, drift_case, drift)
        rows.append({
            'fiscal_year': int(final.fiscal_year),
            'em_score': final.em_score,
            'em_rating_equivalent': final.em_rating_equivalent,
            'crisil_pd_category': category,
            'crisil_pd_raw': raw_pd,
            'target_pd': target_pd,
            'target_pd_bp': target_pd * 10000,
            'target_dd': case.target_dd,
            'equity_value': main.equity_value,
            'debt': main.debt,
            'risk_free_rate': main.risk_free_rate,
            'physical_drift': main.physical_drift,
            **solved,
        })
    return pd.DataFrame(rows)


def score_figure(scores):
    specs = [
        ('z_score', 'Original Altman Z score', 'altman_z_score.png'),
        ('z_double_prime', 'Altman Z″ score', 'altman_z_double_prime.png'),
        ('em_score', 'Altman EM score', 'altman_em_score.png'),
    ]
    folder = ROOT / 'outputs/figures'
    folder.mkdir(parents=True, exist_ok=True)
    for column, title, filename in specs:
        fig, ax = plt.subplots(figsize=(7.5, 4.5))
        values = scores[column]
        ax.plot(scores.fiscal_year, values, marker='o', linewidth=2.2)
        ax.annotate(
            f'FY2026: {values.iloc[-1]:.2f}',
            (scores.fiscal_year.iloc[-1], values.iloc[-1]),
            xytext=(-4, 10), textcoords='offset points', ha='right', fontsize=9,
        )
        ax.set(title=f'Biocon {title}, FY2022_FY2026', xlabel='Fiscal year',
               ylabel='Score', xticks=YEARS)
        ax.grid(alpha=.2)
        fig.tight_layout()
        fig.savefig(folder / filename, dpi=180)
        plt.show()


def main():
    scores = build_scores()
    reverse = reverse_engineer(scores)

    assert len(scores) == 5
    assert scores.fiscal_year.tolist() == YEARS
    assert np.isfinite(scores.select_dtypes('number')).all().all()
    assert len(reverse) == 2
    assert reverse.fsolve_ier.eq(1).all()
    assert reverse.max_scaled_residual.max() < 1e-8
    assert reverse.check_success.all()
    assert reverse.asset_value_check_difference.abs().max() < 0.001
    assert reverse.sigma_v_check_difference.abs().max() < 1e-8

    folder = ROOT / 'data/processed'
    scores.to_csv(folder / 'altman_scores.csv', index=False)
    reverse.to_csv(folder / 'altman_reverse_engineered_volatility.csv', index=False)
    score_figure(scores)

    print('\nALTMAN SCORES')
    print(scores[[
        'fiscal_year', 'z_score', 'z_zone', 'z_double_prime',
        'z_double_prime_zone', 'em_score', 'em_zone',
        'em_rating_equivalent',
    ]].to_string(index=False))
    final = scores.iloc[-1]
    print(
        f'\nFY2026 INTERPRETATION: Original Z is {final.z_zone}, '
        f'Z-double-prime is {final.z_double_prime_zone}, and EM is '
        f'{final.em_zone}. The EM rating equivalent is '
        f'{final.em_rating_equivalent}.'
    )
    print('\nFY2026 EM RATING, PD AND REVERSE-ENGINEERED VOLATILITY')
    print(reverse[[
        'em_score', 'em_rating_equivalent', 'crisil_pd_category',
        'target_pd_bp', 'target_dd', 'drift_case', 'sigma_v', 'sigma_e',
    ]].to_string(index=False))


if __name__ == '__main__':
    main()

In [ ]:
%%writefile src/part_e_comparison.py
"""Part E: compare the five default-risk routes on 30 September 2026."""
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


ROOT = Path(__file__).resolve().parents[1]
DATA = ROOT / 'data/processed'
FIGURES = ROOT / 'outputs/figures'


def final_row(filename, drift='risk_neutral'):
    frame = pd.read_csv(DATA / filename)
    if 'drift_case' in frame:
        frame = frame.loc[frame.drift_case.eq(drift)]
    return frame.iloc[-1]


def build_table():
    iv = final_row('merton_results.csv')
    garch = final_row('garch_merton_results.csv')
    ml = final_row('ml_merton_results.csv')
    rating = pd.read_csv(DATA / 'rating_reverse_engineered_volatility.csv')
    rating = rating.query("drift_case == 'risk_neutral' and case == 'rating_point'").iloc[0]
    altman = final_row('altman_reverse_engineered_volatility.csv')

    rows = [
        ('A_ATM_implied_volatility', iv.sigma_e, iv.sigma_v,
         iv.dd_risk_neutral, iv.pd_risk_neutral_bp,
         'Risk_neutral', 'Point_in_time'),
        ('B_Student_t_GARCH', garch.sigma_e, garch.sigma_v,
         garch.dd_risk_neutral, garch.pd_risk_neutral_bp,
         'Risk_neutral', 'Point_in_time'),
        ('B_Random_Forest', ml.sigma_e, ml.sigma_v,
         ml.dd_risk_neutral, ml.pd_risk_neutral_bp,
         'Risk_neutral', 'Point_in_time'),
        ('C_CRISIL_rating', rating.sigma_e, rating.sigma_v,
         rating.target_dd, rating.target_pd_bp,
         'Risk_neutral_reverse_solve', 'Through_the_cycle'),
        ('D_Altman_EM', altman.sigma_e, altman.sigma_v,
         altman.target_dd, altman.target_pd_bp,
         'Risk_neutral_reverse_solve', 'Accounting_point_in_time'),
    ]
    columns = [
        'route', 'sigma_e', 'sigma_v', 'distance_to_default', 'pd_bp',
        'measure_basis', 'risk_perspective',
    ]
    return pd.DataFrame(rows, columns=columns)


def route_gaps(table):
    values = table.set_index('route')
    return pd.DataFrame({
        'comparison': [
            'ATM_IV_minus_GARCH_sigma_e',
            'ATM_IV_minus_Random_Forest_sigma_e',
            'CRISIL_minus_ATM_IV_sigma_e',
            'Altman_minus_ATM_IV_sigma_e',
        ],
        'difference': [
            values.loc['A_ATM_implied_volatility', 'sigma_e']
            - values.loc['B_Student_t_GARCH', 'sigma_e'],
            values.loc['A_ATM_implied_volatility', 'sigma_e']
            - values.loc['B_Random_Forest', 'sigma_e'],
            values.loc['C_CRISIL_rating', 'sigma_e']
            - values.loc['A_ATM_implied_volatility', 'sigma_e'],
            values.loc['D_Altman_EM', 'sigma_e']
            - values.loc['A_ATM_implied_volatility', 'sigma_e'],
        ],
    })


def figures(table):
    FIGURES.mkdir(parents=True, exist_ok=True)
    labels = ['ATM IV', 'GARCH', 'Random Forest', 'CRISIL', 'Altman EM']
    x = np.arange(len(table))

    fig, ax = plt.subplots(figsize=(8.5, 4.8))
    colours = ['#2563eb', '#f97316', '#16a34a', '#dc2626', '#7c3aed']
    ax.bar(x, table.sigma_e * 100, color=colours)
    ax.set(xticks=x, xticklabels=labels, ylabel='Annualised equity volatility (%)',
           title='Part E_equity volatility comparison')
    ax.grid(axis='y', alpha=.2)
    fig.tight_layout()
    fig.savefig(FIGURES / 'part_e_volatility_comparison.png', dpi=180)
    plt.show()

    fig, ax = plt.subplots(figsize=(8.5, 4.8))
    ax.bar(x, table.pd_bp, color=colours)
    ax.axhline(3, color='#111827', linestyle='--', linewidth=1.3,
               label='Assignment floor_3 bp')
    ax.set(xticks=x, xticklabels=labels, ylabel='Probability of default (bp)',
           title='Part E_reported default probability comparison')
    ax.grid(axis='y', alpha=.2)
    ax.legend()
    fig.tight_layout()
    fig.savefig(FIGURES / 'part_e_pd_comparison.png', dpi=180)
    plt.show()


def main():
    table = build_table()
    gaps = route_gaps(table)
    assert len(table) == 5
    assert np.isfinite(table.select_dtypes('number')).all().all()
    assert table.pd_bp.ge(3 - 1e-10).all()
    table.to_csv(DATA / 'part_e_summary_table.csv', index=False)
    gaps.to_csv(DATA / 'part_e_route_gaps.csv', index=False)
    figures(table)
    display = table.copy()
    display[['sigma_e', 'sigma_v']] *= 100
    print('\nPART E_SUMMARY TABLE')
    print(display.to_string(index=False, formatters={
        'sigma_e': '{:.3f}%'.format,
        'sigma_v': '{:.3f}%'.format,
        'distance_to_default': '{:.4f}'.format,
        'pd_bp': '{:.3f}'.format,
    }))
    print('\nROUTE GAPS')
    print(gaps.to_string(index=False))


if __name__ == '__main__':
    main()

## Data preparation and option-chain cleaning

Biocon's supplied adjusted prices and adjusted returns are used directly; they are not adjusted a second time. Market capitalisation uses the contemporaneous published close. The single missing India VIX value on the 2 March 2024 special Saturday session is carried forward from the latest prior observation.

For each option valuation date, the near-month expiry is retained when more than seven calendar days remain; otherwise the next expiry is used. Zero-volume contracts are removed, moneyness is restricted to `0.90 <= K/S <= 1.10`, and contracts violating the BSM no-arbitrage lower bound are excluded. The upper bound remains a reported diagnostic. Every exclusion is traceable.

In [ ]:
%run src/preprocess_market_inputs.py

## Part A: implied volatility and Merton structural model

### Black_Scholes_Merton implied volatility

For calls and puts with continuous dividend yield `q`:

\[
c=S e^{-qT}N(d_1)-K e^{-rT}N(d_2), \qquad
p=K e^{-rT}N(-d_2)-S e^{-qT}N(-d_1)
\]

\[
d_1=\frac{\ln(S/K)+(r-q+\tfrac12\sigma^2)T}{\sigma\sqrt{T}},
\qquad d_2=d_1-\sigma\sqrt{T}.
\]

The authoritative solver is Newton_Raphson starting from 30%, with bisection on `[1%, 300%]` as fallback. Each solution must reprice within INR 0.0001. `py_vollib` is check_only, as required by the assignment. Call and put ATM IVs are interpolated separately between strikes bracketing `K/S=1` and then averaged.

### Merton equations

Equity is treated as a European call on firm assets:

\[
E=VN(d_1)-De^{-rT}N(d_2), \qquad
\sigma_E E=N(d_1)\sigma_V V.
\]

The default point is `D = short_term_debt + 0.5 x long_term_debt`, `T=1`, and equity value is close price times dated shares outstanding. SciPy `fsolve` jointly estimates `V` and `sigma_V`, starting at `V_0=E+D` and `sigma_V0=sigma_E E/(E+D)`. Every solution is substituted back into both equations and checked independently with the assignment-listed `ItsSypher/merton` package.

Risk_neutral distance to default uses `mu=r`; the physical case uses the annualised historical mean adjusted log return. Reported PD is subject to the mandatory 3 bp floor.

In [ ]:
%run src/implied_volatility.py

In [ ]:
%run src/merton_model.py

### Part A interpretation

On 30 September 2026, ATM IV is approximately **24.915%**, solved asset volatility is **21.653%**, and risk_neutral DD is **9.281**. Raw PD is below the reporting threshold, so PD is shown at the assignment floor of **3 bp**. This high DD does not mean Biocon has no borrowings; it reflects market-implied asset value being large relative to the KMV default point. Observed `D/V` is about 13.9% at the main date.

August 2026 remains a disclosed data-quality concern because call and put ATM IV differ materially. No unassigned discretionary filter is applied to conceal that observation.

## Part B: volatility forecasting

### GARCH(1,1)

Daily adjusted log returns follow:

\[
\sigma_t^2=\omega+\alpha u_{t-1}^2+\beta\sigma_{t-1}^2.
\]

Normal and Student_t errors are estimated separately on the previous 1,000 trading days at each valuation date. Student_t is selected because it has lower BIC on all 12 dates. Reported diagnostics include standard errors, persistence, long_run variance, half_life, likelihood, AIC, BIC, ARCH_LM, and Ljung_Box tests on standardised and squared standardised residuals.

The assignment annualisation uses expected average variance over 252 days:

\[
\bar{\sigma}^2(T)=V_L+\frac{1-e^{-aT}}{aT}[V(0)-V_L],
\qquad \sigma_E^{GARCH}=\sqrt{252\bar{\sigma}^2(T)}.
\]

In [ ]:
%run src/garch_model.py

### Random Forest

The target is log annualised realised volatility over both 21 and 252 future trading days:

\[
RV_{t,h}=\sqrt{\frac{252}{h}\sum_{i=1}^{h}u_{t+i}^2}.
\]

Features known at time `t` are realised volatility over 5, 21 and 63 days; returns over 1, 5 and 21 days; GARCH volatility; India VIX; Nifty 50 21-day return; and a results-month dummy. Training, validation and test periods are chronological 60%/20%/20%, and overlapping forward labels are purged at split boundaries. Hyperparameters are selected on validation data only. Equal weighting is the locked specification; recency weighting is retained only as sensitivity analysis.

Random Forest is used because the sample is modest, nonlinear interactions are plausible, ensemble averaging is robust, and feature importance is directly reportable. The final test result is reported honestly even when GARCH performs better.

In [ ]:
%run src/ml_volatility.py

### Part B interpretation

At the main date, Student_t GARCH gives `sigma_E` about **32.690%**, `sigma_V` **28.410%**, and DD **7.014**. Random Forest gives `sigma_E` about **31.480%**, `sigma_V` **27.359%**, and DD **7.295**. Both raw PDs remain below 3 bp.

The 252-day Random Forest does **not** beat GARCH on the untouched test sample: its GARCH-relative out_of_sample R_squared is negative. This is a substantive result, not a reason to revise the model after observing the test period.

## Part C: rating-implied PD

Biocon's CRISIL AA+ Stable rating maps to the CRISIL study's AA category. The published one-year rates are AAA 0.00%, AA 0.02%, and A 0.07%. The assignment floor raises the point and better category to 0.03%. Geometric adjacent-category boundaries produce a point/lower edge of 3 bp and an upper edge of `sqrt(3 x 7) = 4.5826` bp.

For each target PD, `DD=-N^{-1}(PD)`. Holding 30 September 2026 equity, debt, rate and horizon fixed, the equity equation and target-DD equation are jointly reverse-solved for asset value and asset volatility. The resulting equity volatility is then derived from the Merton volatility relation.

In [ ]:
%run src/rating_model.py

### Part C interpretation

The risk_neutral CRISIL-consistent equity-volatility band is approximately **63.124% to 65.024%**, well above both observed ATM IV and GARCH. This does not prove that one route is wrong. The structural market routes are point_in_time, while agency ratings and historical rating transition/default studies are more through_the_cycle and incorporate qualitative and prospective considerations.

## Part D: Altman accounting models

The original listed-manufacturer score is:

\[
Z=1.2X_1+1.4X_2+3.3X_3+0.6X_4+X_5.
\]

For the non_US/emerging-market comparison:

\[
Z''=6.56X_1+3.26X_2+6.72X_3+1.05X_4, \qquad EM=Z''+3.25.
\]

`X1` is working capital/total assets; `X2` is reserves and surplus/total assets, implemented as Other equity under the assignment's Ind AS instruction; `X3` is EBIT/total assets with EBIT defined consistently as PBT plus finance cost; `X4` uses market equity/total liabilities for original Z and book equity/total liabilities for Z_double_prime; `X5` is sales/total assets.

EM is the preferred accounting model for Biocon because it is the emerging-market specification. The unrounded EM score is mapped through Altman's rating-equivalent intervals and then to the same CRISIL default study used in Part C.

In [ ]:
%run src/altman_model.py

### Part D interpretation

For FY2026, original Z is approximately **2.428, grey**, Z_double_prime is **3.597, safe**, and EM is **6.847, safe**. EM maps to an A equivalent, four notches below the observed AA+ agency rating. The corresponding 7 bp PD gives reverse-solved `sigma_E` about **67.098%**, `sigma_V` **58.319%**, and DD **3.195** under risk_neutral drift.

The difference is economically plausible: Altman reacts mechanically to accounting ratios and weak earnings, whereas the agency assessment also includes liquidity, prospective business conditions, qualitative factors and possible group support.

## Part E: synthesis

The final table places every route on the same 30 September 2026 page while preserving its interpretation. For the structural routes, PD is the reported floored value. Rating and accounting volatilities are reverse-solved values required to make Merton consistent with their target PDs.

In [ ]:
%run src/part_e_comparison.py

In [ ]:
import pandas as pd
from IPython.display import display, Image

summary = pd.read_csv('data/processed/part_e_summary_table.csv')
formatted = summary.copy()
formatted['sigma_e'] = (100 * formatted.sigma_e).map(lambda x: f'{x:.3f}%')
formatted['sigma_v'] = (100 * formatted.sigma_v).map(lambda x: f'{x:.3f}%')
formatted['distance_to_default'] = formatted.distance_to_default.map(lambda x: f'{x:.4f}')
formatted['pd_bp'] = formatted.pd_bp.map(lambda x: f'{x:.3f}')
display(formatted)
display(Image('outputs/figures/part_e_volatility_comparison.png'))
display(Image('outputs/figures/part_e_pd_comparison.png'))

## Overall conclusion

The models agree directionally that Biocon's one-year default risk is low at the valuation date, but they disagree substantially about the volatility consistent with that conclusion.

- ATM IV is the lowest volatility route and produces the highest DD.
- GARCH and Random Forest are higher than ATM IV but still produce raw PD below the reporting floor.
- The CRISIL historical rating calibration requires much higher volatility to reach even its floored rating PD.
- Altman EM is the most conservative route, mapping FY2026 accounting ratios to A and 7 bp.

The ATM_IV minus GARCH volatility gap is negative, approximately **minus 7.78 percentage points**. It may be discussed as a snapshot volatility-risk-premium proxy, but only cautiously because option IV and the one-year GARCH forecast use different data, market microstructure and expectations.

### Limitations

1. The 3 bp floor masks differences among very small structural PD estimates.
2. Merton assumes one zero-coupon debt issue, constant asset volatility and default only at maturity.
3. Option settlement prices and sparse contracts can distort the smile, especially August 2026.
4. GARCH leaves some short-lag mean autocorrelation even though squared-residual diagnostics are satisfactory.
5. Random Forest underperforms GARCH out of sample and should not be presented as superior.
6. Rating and accounting routes are calibrated to different concepts and horizons from market-implied structural PD.

Detailed company filing, RBI, NSE, CRISIL and repository citations should be listed in the accompanying technical report appendix.